# ManifoldEM on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flatironinstitute/ManifoldEM/blob/main/tutorial/ManifoldEM_Colab.ipynb)

[ManifoldEM](https://github.com/flatironinstitute/ManifoldEM) recovers **continuous conformational changes** of a
macromolecule from single-particle cryo-EM images. It analyses the particles of each projection direction (PD) with
diffusion maps and nonlinear Laplacian spectral analysis (NLSA), stitches the directions together into a
conformational coordinate, and reconstructs a **free-energy profile** and a **3D movie** of the motion.

This notebook runs the complete pipeline in the browser, with no local installation and no GUI. It is
equivalent to the [command-line tutorial](https://github.com/flatironinstitute/ManifoldEM/blob/main/tutorial/README.md),
with viewers for every decision the GUI would normally help you make.

**Quick start:** choose **Runtime ▸ Run all**. With the default settings the notebook builds a small synthetic
dataset with a *known* motion and takes it through every step in about 10–15 minutes on the free tier. Then
re-run it with your own data.

| Section | What happens | ManifoldEM step | Time (synthetic demo, 2 CPUs) |
|---|---|---|---|
| **0 · Setup** | install ManifoldEM in its own environment | – | 2–4 min |
| **1 · Data** | synthetic demo, RyR1 tutorial data or your own STAR + MRCS | – | ~1 min |
| **2 · Project** | create (or re-open) a project | `init` | seconds |
| **3 · Projection directions** | bin particles by orientation, set occupancy thresholds | `threshold` | seconds |
| **4 · Manifold embedding** | distances, diffusion maps, NLSA, movies for every PD | `calc-distance` … `nlsa-movie` | 3–5 min |
| **5 · Explore** | browse PDs: class averages, eigenvalues, NLSA movies | (GUI *Eigenvectors* tab) | – |
| **6 · Anchors** | pick reference PDs and the direction of the motion | (GUI *Set PD Anchors*) | – |
| **7 · Coordinate & landscape** | belief propagation, free-energy profile, per-state particle stacks | `find-ccs`, `probability-landscape`, `trajectory` | ~1 min |
| **8 · Volumes** | reconstruct and view a 3D movie of the conformational change | `utility mrcs2mrc` | ~1 min |
| **9 · Python API** | run your own code against ManifoldEM's `data_store` | – | – |
| **10 · Save** | checkpoint to Google Drive, download results | – | – |

**How the notebook works**
* Every step is a cell with a small form. Change values in the form, not in the code. Each cell can be re-run.
* PD numbers and ψ (eigenvector) numbers start at **1**, as in the GUI and in the output folders `topos/PrD_<n>/psi_<k>.gif`.
* After a disconnect or a runtime restart, run section 0 again and then **2.2 Re-open a project**. Results on the
  Colab disk survive a restart, but not a disconnect. Section 10 saves checkpoints to Google Drive.
* **Runtime:** ManifoldEM is CPU-only, so a GPU runtime does not help. The free tier has 2 CPUs and ~12 GB of RAM,
  which is enough for the demo and for small or downsampled datasets. For full-size data, use a high-RAM / more-CPU
  runtime (Colab Pro) or a workstation (see the main README).

*Please cite:* A. Ojha *et al.*, *The ManifoldEM method for cryo-EM: a step-by-step breakdown accompanied by a modern
Python implementation*, Acta Cryst. D81, 89–104 (2025), [doi:10.1107/S2059798325001469](https://doi.org/10.1107/S2059798325001469).

## 0 · Setup

In [ ]:
#@title 0.1 · Check the runtime { display-mode: "form" }
import os, sys, shutil, platform, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
ROOT = Path('/content/manifoldem') if IN_COLAB else Path.cwd() / 'manifoldem_colab'
ENV_DIR = ROOT / 'env'          # ManifoldEM's own Python environment
ROOT.mkdir(parents=True, exist_ok=True)

try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / 1e9
except ImportError:
    RAM_GB = float('nan')
print(f"Python {platform.python_version()} · {os.cpu_count()} CPUs · {RAM_GB:.1f} GB RAM · "
      f"{shutil.disk_usage(ROOT).free / 1e9:.0f} GB free disk")
print(f"Working folder: {ROOT}")
if shutil.which('nvidia-smi'):
    print("ℹ️  A GPU is attached, but ManifoldEM only uses CPUs (Runtime ▸ Change runtime type ▸ CPU is fine).")
if (os.cpu_count() or 1) <= 2:
    print("ℹ️  2 CPUs: fine for the demo and small datasets; a high-RAM runtime with more CPUs is much faster for real data.")

In [ ]:
#@title 0.2 · Install ManifoldEM (≈ 2–4 min, once per session) { display-mode: "form" }
#@markdown ManifoldEM is installed from GitHub into its own Python 3.12 environment, because it needs NumPy < 2
#@markdown while Colab runs NumPy 2. Colab's own packages are left untouched, so **no runtime restart is needed**.
MANIFOLDEM_REPO = "https://github.com/flatironinstitute/ManifoldEM"  #@param {type:"string"}
MANIFOLDEM_VERSION = "v0.4.0"  #@param {type:"string"}
#@markdown `MANIFOLDEM_VERSION` is a tag, branch or commit (`main` = latest development version). Use your fork's URL to test your own changes.
REINSTALL = False  #@param {type:"boolean"}
import json, time, importlib.util

SUBPROCESS_ENV = dict(os.environ, MPLBACKEND='Agg')   # the kernel's inline backend does not exist in ManifoldEM's env

def _run(cmd, quiet=True):
    r = subprocess.run([str(c) for c in cmd], capture_output=quiet, text=True, env=SUBPROCESS_ENV)
    if r.returncode:
        print((r.stdout or '')[-3000:], (r.stderr or '')[-3000:])
        raise RuntimeError(f"failed: {' '.join(map(str, cmd))}")
    return r.stdout if quiet else ''

ref = MANIFOLDEM_VERSION.strip()
SPEC = f"manifoldem @ git+{MANIFOLDEM_REPO.strip().rstrip('/')}" + (f"@{ref}" if ref else "")
PY = ENV_DIR / 'bin' / 'python'
marker = ENV_DIR / '.installed.json'
if marker.exists() and json.loads(marker.read_text()).get('spec') == SPEC and not REINSTALL:
    INSTALLED = json.loads(marker.read_text())
    print(f"✔ already installed: ManifoldEM {INSTALLED['manifoldem']} ({SPEC})")
else:
    t0 = time.time()
    uv = [shutil.which('uv')] if shutil.which('uv') else None
    if uv is None:
        _run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
        uv = [sys.executable, '-m', 'uv']
    if ENV_DIR.exists():
        shutil.rmtree(ENV_DIR)
    print('· creating a Python 3.12 environment ...')
    _run(uv + ['venv', '--seed', '--python', '3.12', ENV_DIR])
    print('· installing the scientific stack (NumPy < 2) ...')
    _run(uv + ['pip', 'install', '--python', PY, 'numpy<2', 'scipy', 'numba', 'pandas', 'scikit-learn',
               'scikit-image', 'h5py', 'imageio', 'mrcfile', 'nptyping', 'fasthog', 'toml', 'tqdm',
               'matplotlib', 'threadpoolctl', 'setuptools', 'setuptools-scm'])
    print(f'· installing {SPEC} ...')
    _run([PY, '-m', 'pip', 'install', '-q', '--no-warn-conflicts', '--progress-bar', 'off', 'numpy<2', SPEC])
    check = ("import json, numpy, cv2, ManifoldEM, ManifoldEM.calc_distance, ManifoldEM.find_conformational_coords;"
             "o = cv2.optflow; [getattr(o, n) for n in ('createOptFlow_DeepFlow', 'calcOpticalFlowDenseRLOF')];"
             "print(json.dumps(dict(manifoldem=ManifoldEM.__version__, numpy=numpy.__version__, cv2=cv2.__version__)))")
    try:
        out = _run([PY, '-c', check])
    except RuntimeError:
        # opencv-python-headless and opencv-contrib-python share one cv2/ folder; the contrib build must win
        ver = _run([PY, '-c', 'import importlib.metadata as m; print(m.version("opencv-contrib-python"))']).strip()
        _run([PY, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'opencv-contrib-python=={ver}'])
        out = _run([PY, '-c', check])
    INSTALLED = dict(json.loads(out.strip().splitlines()[-1]), spec=SPEC)
    marker.write_text(json.dumps(INSTALLED))
    print(f"✔ ManifoldEM {INSTALLED['manifoldem']} installed in {(time.time() - t0) / 60:.1f} min "
          f"(numpy {INSTALLED['numpy']}, OpenCV {INSTALLED['cv2']})")
if importlib.util.find_spec('mrcfile') is None:   # small, NumPy-2 compatible; used to display volumes
    _run([sys.executable, '-m', 'pip', 'install', '-q', 'mrcfile'])
subprocess.run([str(ENV_DIR / 'bin' / 'manifold-cli')], capture_output=True, env=SUBPROCESS_ENV)  # warm-up (first import compiles bytecode)

In [ ]:
#@title 0.3 · Write the ManifoldEM bridge script { display-mode: "form" }
#@markdown Small helper that runs inside ManifoldEM's environment and answers the notebook's questions (PD
#@markdown summaries, anchors, input checks, ...). Expand the cell to read it.
BRIDGE_SOURCE = r'''"""Bridge between the Colab notebook and ManifoldEM.

ManifoldEM needs numpy<2, while Colab's own Python runs numpy 2, so the notebook keeps ManifoldEM in a
separate environment and talks to it through this script:

    <env>/bin/python mem_bridge.py <command> <args.json> <result.json>

Each command reads keyword arguments from args.json and writes a JSON result (plus, for some
commands, an .npz file of arrays next to it).  Nothing here is needed when you use ManifoldEM from a
normal terminal; it only wraps the documented `params` / `data_store` API and works around a few
sharp edges (see the notes on each command).
"""
import glob
import inspect
import json
import os
import re
import shutil
import sys
import time

import numpy as np

COMMANDS = {}


class UserError(Exception):
    """A problem with the user's input or project state (reported without a traceback)."""


def command(func):
    COMMANDS[func.__name__.replace('_', '-')] = func
    return func


def _load(toml):
    from ManifoldEM.params import params
    params.load(os.path.abspath(toml))  # note: chdir()s into the project folder
    return params


def _prds():
    from ManifoldEM.data_store import data_store
    return data_store.get_prds()


def _cc_bug():
    """True if belief propagation reads Anchor.CC as 0-based (ManifoldEM <= 0.4.0, see find_ccs)."""
    from ManifoldEM.CC import runGlobalOptimization
    return '[anch_list[anchor_id].CC, index]' in inspect.getsource(runGlobalOptimization)


# --------------------------------------------------------------------------------------------- project
_A, _B = 987654, 876543  # sentinels used to turn ManifoldEM's path functions into templates


def _paths(params):
    def tmpl(path):
        return os.path.abspath(path).replace(str(_A), '{p}').replace(str(_B), '{k}')

    topo_dir = os.path.dirname(tmpl(params.get_psi_gif(_A, _B)))
    return dict(
        work=os.getcwd(), toml=os.path.abspath(params.proj_file), out_dir=os.path.abspath(params.out_dir),
        pd_file=os.path.abspath(params.pd_file), CC_dir=os.path.abspath(params.CC_dir),
        CC_file=os.path.abspath(params.CC_file), CC_OF_dir=os.path.abspath(params.CC_OF_dir),
        traj=os.path.abspath(f'{params.traj_file}name{params.traj_name}.pkl'),
        traj_vars=os.path.abspath(f'{params.traj_file}name{params.traj_name}_vars.pkl'),
        OM_file=os.path.abspath(params.OM_file), bin_dir=os.path.abspath(params.bin_dir),
        vols_dir=os.path.abspath(params.postproc_mrcs2mrc_dir),
        denoise_dir=os.path.abspath(params.postproc_denoise_dir),
        # templates: {p} = 0-based PD index unless noted, {k} = 0-based psi index unless noted
        dist=tmpl(params.get_dist_file(_A)), psi=tmpl(params.get_psi_file(_A)),
        psi2=tmpl(params.get_psi2_file(_A, _B)), EL=tmpl(params.get_EL_file(_A)),
        gif=tmpl(params.get_psi_gif(_A, _B)),  # 1-based PD and psi
        topo=tmpl(params.get_topos_path(_A, _B)),  # 1-based PD and psi
        class_avg=os.path.join(topo_dir, 'class_avg.png'),  # 1-based PD
        eig_spec=os.path.join(topo_dir, 'eig_spec.txt'),  # 1-based PD
    )


def _status(params, n_pd):
    """Which pipeline steps have complete outputs on disk."""
    P = _paths(params)
    pds = range(n_pd)
    have = lambda f: os.path.isfile(f)
    done = dict(
        init=True,
        binning=have(P['pd_file']),
        calc_distance=n_pd > 0 and all(have(P['dist'].format(p=p)) for p in pds),
        manifold_analysis=n_pd > 0 and all(have(P['psi'].format(p=p)) for p in pds),
        psi_analysis=n_pd > 0 and all(have(P['psi2'].format(p=p, k=k)) for p in pds for k in range(params.num_psi)),
        nlsa_movie=n_pd > 0 and all(have(P['class_avg'].format(p=p + 1)) for p in pds),
        find_ccs=have(P['CC_file']),
        probability_landscape=have(P['traj']),
        trajectory=len(glob.glob(os.path.join(P['bin_dir'], '*.star'))) > 0,
        volumes=len(glob.glob(os.path.join(P['vols_dir'], '*.mrc'))),
    )
    return done


@command
def summary(toml, out_npz=None, n_particles_show=6000):
    """Everything the notebook plots about the projection directions (bins the data on first use)."""
    import ManifoldEM
    params = _load(toml)
    prds = _prds()  # builds pd_data.pkl the first time (this is the "binning" step)
    n_pd = prds.n_thresholded
    occ = np.asarray(prds.occupancy, dtype=int)
    rng = np.random.default_rng(0)
    pos_full = prds.pos_full
    sel = rng.choice(pos_full.shape[1], min(n_particles_show, pos_full.shape[1]), replace=False)
    anchors = [[int(k), int(a.CC), int(a.sense.value)] for k, a in sorted(prds.anchors.items())]
    trash = sorted(int(t) for t in prds.trash_ids)
    edges = np.asarray(prds.neighbor_graph.get('Edges', np.zeros((0, 2))), dtype=int).reshape(-1, 2)
    if out_npz:
        np.savez(out_npz, occupancy=occ, n_used=np.minimum(occ, prds.thres_high),
                 cluster_ids=np.asarray(prds.cluster_ids, dtype=int), thres_ids=np.asarray(prds.thres_ids, dtype=int),
                 pos_pd=np.asarray(prds.pos_thresholded, dtype=float),
                 theta=np.asarray(prds.theta_thresholded, dtype=float),
                 phi=np.asarray(prds.phi_thresholded, dtype=float),
                 bin_centers=np.asarray(prds.bin_centers, dtype=float),
                 occupancy_full=np.asarray(prds.occupancy_full, dtype=int),
                 pos_particles=np.asarray(pos_full[:, sel], dtype=float), edges=edges,
                 anchors=np.asarray(anchors, dtype=int).reshape(-1, 3), trash=np.asarray(trash, dtype=int))
    return dict(
        project=params.project_name, version=getattr(ManifoldEM, '__version__', '?'),
        level=params.project_level.name, n_pd=int(n_pd), n_bins=int(prds.n_bins),
        n_particles=int(pos_full.shape[1]), n_used=int(np.minimum(occ, prds.thres_high).sum()),
        n_clusters=int(len(set(np.asarray(prds.cluster_ids).tolist()))),
        thres_low=int(prds.thres_low), thres_high=int(prds.thres_high),
        aperture_index=int(params.aperture_index), shannon_angle=float(params.sh), ang_width=float(params.ang_width),
        tess_type=params.tess_hemisphere_type, hemisphere_vec=[float(v) for v in params.tess_hemisphere_vec],
        box=int(params.ms_num_pixels), pixel_size=float(params.ms_pixel_size),
        diameter=float(params.particle_diameter), resolution=float(params.ms_estimated_resolution),
        num_psi=int(params.num_psi), num_psi_truncated=int(params.num_psi_truncated),
        states_per_coord=int(params.states_per_coord), con_order_range=int(params.con_order_range),
        width_1D=int(params.width_1D), traj_name=str(params.traj_name), ncpu=int(params.ncpu),
        nlsa_fps=float(params.nlsa_fps), nlsa_tune=int(params.nlsa_tune),
        distance_filter_type=params.distance_filter_type,
        anchors=anchors, trash=trash, cc_bug=_cc_bug(), paths=_paths(params), done=_status(params, n_pd),
        files=dict(star=params.align_param_file, stack=params.img_stack_file, volume=params.avg_vol_file,
                   mask=params.mask_vol_file),
    )


@command
def set_params(toml, values):
    """Persist parameter changes in the project's .toml.

    (`manifold-cli <step> --flag value` prints "Changing param ..." but every step reloads the .toml,
    so only `init` and `threshold` flags actually stick. Editing the .toml is the reliable way.)"""
    params = _load(toml)
    cls = type(params)
    changed = {}
    for key, value in values.items():
        if not hasattr(cls, key):
            raise UserError(f'unknown ManifoldEM parameter: {key}')
        current = getattr(params, key)
        if isinstance(current, bool):
            value = bool(value)
        elif isinstance(current, (int, float, str)):
            value = type(current)(value)
        if value != current:
            changed[key] = [current, value]
            setattr(params, key, value)
    params.save()
    return dict(changed=changed)


@command
def bin_particles(toml, low, high, aperture_index=0, tess_type=''):
    """Set the PD occupancy thresholds (and optionally aperture/tessellation) and (re)bin."""
    from ManifoldEM.params import ProjectLevel
    params = _load(toml)
    pkl_before = os.path.getmtime(params.pd_file) if os.path.isfile(params.pd_file) else None
    geometry_changed = False
    if aperture_index and int(aperture_index) != params.aperture_index:
        params.aperture_index, geometry_changed = int(aperture_index), True
    if tess_type and tess_type != params.tess_hemisphere_type:
        params.tess_hemisphere_type, geometry_changed = tess_type, True
    params.prd_thres_low, params.prd_thres_high = int(low), int(high)
    if params.project_level.value < ProjectLevel.BINNING.value:
        params.project_level = ProjectLevel.BINNING
    params.save()
    if geometry_changed and pkl_before is not None:
        os.environ['MANIFOLD_REBUILD_DS'] = '1'  # data_store only notices threshold changes by itself
    _prds()
    rebuilt = pkl_before is None or os.path.getmtime(params.pd_file) != pkl_before
    stale = rebuilt and len(glob.glob(os.path.join(params.dist_dir, '*.h5'))) > 0
    return dict(rebuilt=rebuilt, stale_downstream=stale)


@command
def clean(toml, what):
    """Delete outputs of later steps (after re-binning, or to force a step to be recomputed)."""
    params = _load(toml)
    out = params.out_dir
    groups = dict(
        embedding=[params.dist_dir, params.psi_dir, params.psi2_dir, os.path.join(out, 'topos')],
        movies=glob.glob(os.path.join(out, 'topos', 'PrD_*', 'class_avg.png')),
        cc=[params.CC_dir],
        landscape=glob.glob(os.path.join(out, 'ELConc*')) + [params.traj_dir],
        trajectory=[params.bin_dir],
        volumes=[os.path.join(out, 'postproc')],
    )
    removed = []
    for name in what:
        for path in groups[name]:
            if os.path.isdir(path):
                shutil.rmtree(path)
                removed.append(path)
            elif os.path.isfile(path):
                os.remove(path)
                removed.append(path)
    params.create_dir()  # ManifoldEM expects its folder skeleton to exist
    return dict(removed=len(removed))


@command
def incomplete(toml, step):
    """0-based PDs whose outputs for `step` are missing, for resuming with `--prds`."""
    params = _load(toml)
    P = _paths(params)
    n = params.prd_n_active
    checks = {
        'calc-distance': lambda p: os.path.isfile(P['dist'].format(p=p)),
        'manifold-analysis': lambda p: os.path.isfile(P['psi'].format(p=p))
        and os.path.isfile(P['eig_spec'].format(p=p + 1)),
        'psi-analysis': lambda p: all(os.path.isfile(P['psi2'].format(p=p, k=k)) for k in range(params.num_psi)),
        'nlsa-movie': lambda p: os.path.isfile(P['class_avg'].format(p=p + 1)),
    }
    # `manifold-analysis --prds` only creates topos/PrD_1..PrD_len(prds), so create all of them here
    for p in range(n):
        os.makedirs(os.path.dirname(P['class_avg'].format(p=p + 1)), exist_ok=True)
    return dict(prds=[p for p in range(n) if not checks[step](p)], n_pd=n)


@command
def remove_movies(toml, prds=None):
    """nlsa-movie skips PDs that already have class_avg.png, so delete it to force fresh movies."""
    params = _load(toml)
    P = _paths(params)
    prds = range(params.prd_n_active) if prds is None else prds
    n = 0
    for p in prds:
        f = P['class_avg'].format(p=p + 1)
        if os.path.isfile(f):
            os.remove(f)
            n += 1
    return dict(removed=n)


# ----------------------------------------------------------------------------- anchors & find-ccs
def _unanchored_components(prds, num_psi):
    """Connected components (after removing trash PDs) that contain no anchor: their PDs get no CC."""
    from copy import deepcopy
    from ManifoldEM.FindCCGraph import prune
    if prds.trash_ids:
        import contextlib, io
        with contextlib.redirect_stdout(io.StringIO()):
            G, _ = prune(deepcopy(prds.neighbor_graph), prds.trash_ids, num_psi)
    else:
        G = prds.neighbor_graph
    missing = []
    for comp in G['NodesConnComp']:
        comp = [int(c) for c in np.atleast_1d(comp)]
        if any(c in prds.trash_ids for c in comp):
            continue
        if not any(c in prds.anchors for c in comp):
            missing.append(comp)
    return missing


@command
def set_anchors(toml, anchors, trash):
    """Replace all anchors and removed ('trash') PDs.

    anchors: list of [pd (0-based), psi (1-based, i.e. psi_<psi>.gif), 'FWD' | 'REV']; trash: 0-based PDs.
    Anchor.CC is stored 1-based, as the GUI does; find_ccs() below handles ManifoldEM's indexing quirk."""
    from ManifoldEM.data_store import Anchor, Sense
    params = _load(toml)
    prds = _prds()
    n = prds.n_thresholded
    trash = sorted({int(t) for t in trash})
    for t in trash:
        if not 0 <= t < n:
            raise UserError(f'PD {t + 1} does not exist (valid: 1..{n})')
    prds.anchors = {}
    for pd0, psi, sense in anchors:
        pd0, psi, sense = int(pd0), int(psi), str(sense).upper()
        if not 0 <= pd0 < n:
            raise UserError(f'PD {pd0 + 1} does not exist (valid: 1..{n})')
        if not 1 <= psi <= params.num_psi:
            raise UserError(f'psi {psi} is out of range (valid: 1..{params.num_psi})')
        if pd0 in trash:
            raise UserError(f'PD {pd0 + 1} cannot be both an anchor and removed')
        if sense not in ('FWD', 'REV'):
            raise UserError(f"sense must be FWD or REV, not {sense!r}")
        prds.insert_anchor(pd0, Anchor(CC=psi, sense=Sense.FWD if sense == 'FWD' else Sense.REV))
    prds.trash_ids = set(trash)
    prds.save()
    missing = _unanchored_components(prds, params.num_psi)
    return dict(n_anchors=len(prds.anchors), n_trash=len(trash),
                unanchored_components=[[c + 1 for c in comp] for comp in missing])


@command
def find_ccs(toml, ncpu=1, bp_num_psi=0):
    """Run find-ccs (optical flow + belief propagation) with two safety nets.

    1. Anchor.CC: the GUI and the "every PD is an anchor" shortcut treat CC as 1-based, but belief
       propagation (CC/runGlobalOptimization.py) indexes states with CC directly, so Anchor(CC=1)
       pins psi_2.  When that bug is present the anchors are shifted by -1 in memory only.
    2. bp_num_psi > 0 limits belief propagation to psi_1..psi_<bp_num_psi> (1 = keep psi_1 everywhere and
       only propagate the sense).  num_psi is restored afterwards.
    Cached optical flow is discarded when it no longer matches the inputs."""
    from ManifoldEM.data_store import Anchor
    from ManifoldEM.find_conformational_coords import op as find_ccs_op
    params = _load(toml)
    num_psi = params.num_psi
    bp_psi = int(bp_num_psi) if bp_num_psi else num_psi
    if not 1 <= bp_psi <= num_psi:
        raise UserError(f'bp_num_psi must be between 1 and num_psi={num_psi}')
    prds = _prds()
    if not prds.anchors:
        raise UserError('No anchors set. Run step 6.2 first (find-ccs would otherwise silently do nothing).')
    for pd0, a in prds.anchors.items():
        if not 1 <= a.CC <= bp_psi:
            raise UserError(f'Anchor on PD {pd0 + 1} uses psi {a.CC}, but belief propagation only considers '
                             f'psi 1..{bp_psi}. Change the anchor or BP_NUM_PSI.')
    # invalidate cached optical flow if the movies changed or the number of psi changed
    psi2 = glob.glob(os.path.join(params.psi2_dir, '*.h5'))
    marker = os.path.join(params.CC_dir, 'notebook_cc_inputs.json')
    stamp = dict(bp_psi=bp_psi, newest_psi2=max((os.path.getmtime(f) for f in psi2), default=0),
                 trash=sorted(int(t) for t in prds.trash_ids))
    old = json.load(open(marker)) if os.path.isfile(marker) else None
    if old != stamp:
        for d in (params.CC_OF_dir, params.CC_meas_dir, params.CC_dir_temp):
            shutil.rmtree(d, ignore_errors=True)
        params.create_dir()
        os.makedirs(params.CC_dir_temp, exist_ok=True)
    bp_path = len(prds.anchors) + len(prds.trash_ids) != prds.n_thresholded
    shifted = bp_path and _cc_bug()
    saved_anchors = dict(prds.anchors)
    if shifted:
        prds.anchors = {k: Anchor(a.CC - 1, a.sense) for k, a in saved_anchors.items()}
    params.ncpu, params.num_psi = int(ncpu), bp_psi
    params.save()  # find-ccs reloads the .toml
    t0 = time.time()
    try:
        find_ccs_op()
    finally:
        prds.anchors = saved_anchors
        params.load(os.path.abspath(params.proj_file))
        params.num_psi = num_psi
        params.save()
    if not os.path.isfile(params.CC_file):
        raise UserError('find-ccs did not write CC_file.pkl; see the log above.')
    with open(marker, 'w') as f:
        json.dump(stamp, f)
    from ManifoldEM import myio
    cc = myio.fin1(params.CC_file)
    psinums, senses = np.asarray(cc['psinums'][0]), np.asarray(cc['senses'][0])
    return dict(seconds=round(time.time() - t0, 1), anchors_shifted=shifted, bp_num_psi=bp_psi,
                n_assigned=int((psinums >= 0).sum()), n_pd=int(len(psinums)))


# ---------------------------------------------------------------------------------------- inputs
REQUIRED_COLUMNS = ['rlnAngleRot', 'rlnAngleTilt', 'rlnAnglePsi', 'rlnDefocusU', 'rlnDefocusV',
                    'rlnVoltage', 'rlnSphericalAberration', 'rlnAmplitudeContrast']


def _read_star(star):
    """Particles table and first optics row, read with ManifoldEM's own (strict) parser."""
    from ManifoldEM.star import parse_star, parse_star_optics
    with open(star) as f:
        new_style = any(line.startswith('data_optics') for line in f)
    if new_style:
        optics, skip = parse_star_optics(star)
        return parse_star(star, skip), optics
    df = parse_star(star, 0)
    return df, df.iloc[:1]


@command
def inspect_inputs(star, stack, n_show=16, out_npz=None):
    """Check a STAR file + particle stack the way ManifoldEM will read them."""
    import mrcfile
    report = dict(problems=[], warnings=[])
    try:
        df, optics = _read_star(star)
    except Exception as exc:  # pragma: no cover - reported to the user
        report['problems'].append(f'ManifoldEM cannot parse the STAR file: {exc!r}')
        return report
    cols = set(df.columns) | set(optics.columns)
    report['columns'] = sorted(cols)
    report['missing'] = [c for c in REQUIRED_COLUMNS if c not in cols]
    if report['missing']:
        report['problems'].append('missing required STAR columns: ' + ', '.join(report['missing']))
    has_origin = {'rlnOriginX', 'rlnOriginY'} <= cols or {'rlnOriginXAngst', 'rlnOriginYAngst'} <= cols
    if not has_origin:
        report['warnings'].append('no origin (shift) columns: images are assumed to be centred')
    with mrcfile.mmap(stack, mode='r', permissive=True) as m:
        shape = m.data.shape
        report['stack_is_image_stack'] = bool(m.is_image_stack())
        report['header_pixel_size'] = float(m.voxel_size.x)
        n_img = shape[0] if len(shape) == 3 else 1
        rng = np.random.default_rng(1)
        idx = np.sort(rng.choice(n_img, min(n_show, n_img), replace=False))
        imgs = np.asarray(m.data[idx], dtype=np.float32) if len(shape) == 3 else np.asarray(m.data)[None]
    report.update(n_particles=int(len(df)), n_images=int(n_img), box=int(shape[-1]))
    if len(shape) != 3 or shape[-1] != shape[-2]:
        report['problems'].append(f'the stack must hold square images, found shape {shape}')
    if n_img != len(df):
        report['problems'].append(f'the STAR file has {len(df)} particles but the stack has {n_img} images '
                                  '(ManifoldEM pairs STAR row i with stack image i and ignores rlnImageName)')
    if not report['stack_is_image_stack']:
        report['warnings'].append('the MRC header does not flag the file as an image stack; ManifoldEM will '
                                  'rewrite the header in place (work on a copy)')
    px = None
    if 'rlnImagePixelSize' in optics.columns:
        px = float(optics['rlnImagePixelSize'].values[0])
    elif {'rlnDetectorPixelSize', 'rlnMagnification'} <= cols:
        px = float(df['rlnDetectorPixelSize'].values[0]) * 1e4 / float(df['rlnMagnification'].values[0])
    report['star_pixel_size'] = px
    for key, col in (('voltage', 'rlnVoltage'), ('cs', 'rlnSphericalAberration'), ('amp_contrast', 'rlnAmplitudeContrast')):
        if col in cols:
            src = optics if col in optics.columns else df
            report[key] = float(src[col].values[0])
    if 'rlnOpticsGroup' in df.columns and df['rlnOpticsGroup'].nunique() > 1:
        report['warnings'].append(f"{df['rlnOpticsGroup'].nunique()} optics groups found: ManifoldEM uses the "
                                  'voltage/Cs/amplitude contrast of the first one for all particles')
    if out_npz is not None and not report['missing']:
        take = np.random.default_rng(2).choice(len(df), min(20000, len(df)), replace=False)
        np.savez(out_npz, images=imgs, image_index=idx,
                 rot=df['rlnAngleRot'].values[take].astype(float), tilt=df['rlnAngleTilt'].values[take].astype(float),
                 psi=df['rlnAnglePsi'].values[take].astype(float),
                 defocus=(0.5 * (df['rlnDefocusU'].values + df['rlnDefocusV'].values)).astype(float))
    return report


@command
def downsample(star, stack, out_dir, factor):
    """Fourier-crop every particle image by `factor` (e.g. 2: 336 px -> 168 px) and write a matching STAR.

    Speeds up every ManifoldEM step roughly by factor**2 to factor**3 and cuts memory by factor**2; the
    new Nyquist limit is 2 x the new pixel size."""
    import mrcfile
    os.makedirs(out_dir, exist_ok=True)
    with mrcfile.mmap(stack, mode='r', permissive=True) as m:
        n_img, n = m.data.shape[0], m.data.shape[-1]
        px_hdr = float(m.voxel_size.x)
        new_n = int(round(n / factor / 2)) * 2
        scale = n / new_n
        out_stack = os.path.join(out_dir, os.path.splitext(os.path.basename(stack))[0] + f'_bin{factor:g}.mrcs')
        with mrcfile.new_mmap(out_stack, shape=(n_img, new_n, new_n), mrc_mode=2, overwrite=True) as out:
            s = n // 2 - new_n // 2
            for i0 in range(0, n_img, 500):
                block = np.asarray(m.data[i0:i0 + 500], dtype=np.float32)
                F = np.fft.fftshift(np.fft.fft2(block), axes=(-2, -1))[:, s:s + new_n, s:s + new_n]
                out.data[i0:i0 + len(block)] = (np.fft.ifft2(np.fft.ifftshift(F, axes=(-2, -1))).real
                                                / scale ** 2).astype(np.float32)
                print(f'\r  downsampled {min(i0 + 500, n_img)}/{n_img} images', end='', flush=True)
            out.set_image_stack()
            if px_hdr > 0:
                out.voxel_size = px_hdr * scale
    print()
    # rewrite the STAR line by line: pixel origins shrink, optics pixel size / box change
    out_star = os.path.join(out_dir, os.path.splitext(os.path.basename(star))[0] + f'_bin{factor:g}.star')
    header, block, lines_out = [], None, []
    stack_name = os.path.basename(out_stack)
    with open(star) as f:
        lines = f.read().splitlines()
    for line in lines:
        s_line = line.strip()
        if s_line.startswith('data_'):
            block, header = s_line, []
        elif s_line.startswith('_rln'):
            header.append(s_line.split()[0][1:])
        elif s_line and header and not s_line.startswith(('#', 'loop_')):
            vals = s_line.split()
            if len(vals) == len(header):
                for j, col in enumerate(header):
                    if col in ('rlnOriginX', 'rlnOriginY'):
                        vals[j] = f'{float(vals[j]) / scale:.6f}'
                    elif col == 'rlnImagePixelSize':
                        vals[j] = f'{float(vals[j]) * scale:.6f}'
                    elif col == 'rlnImageSize':
                        vals[j] = str(new_n)
                    elif col == 'rlnImageName' and '@' in vals[j]:
                        vals[j] = vals[j].split('@')[0] + '@' + stack_name
                line = ' '.join(vals)
        lines_out.append(line)
    with open(out_star, 'w') as f:
        f.write('\n'.join(lines_out) + '\n')
    return dict(star=out_star, stack=out_stack, box=new_n, scale=scale)


# -------------------------------------------------------------------------------- synthetic data
def _relion_matrix(rot, tilt, psi):
    """RELION Euler_angles2matrix (ZYZ, degrees), vectorised -> (m, 3, 3)."""
    a, b, g = (np.deg2rad(np.asarray(v, dtype=float)) for v in (rot, tilt, psi))
    ca, sa, cb, sb, cg, sg = np.cos(a), np.sin(a), np.cos(b), np.sin(b), np.cos(g), np.sin(g)
    A = np.empty(a.shape + (3, 3))
    A[..., 0, 0], A[..., 0, 1], A[..., 0, 2] = cg * cb * ca - sg * sa, cg * cb * sa + sg * ca, -cg * sb
    A[..., 1, 0], A[..., 1, 1], A[..., 1, 2] = -sg * cb * ca - cg * sa, -sg * cb * sa + cg * ca, sg * sb
    A[..., 2, 0], A[..., 2, 1], A[..., 2, 2] = sb * ca, sb * sa, cb
    return A


@command
def synthetic(src, out, n_particles=6000, box=64, levels=20, amp_angst=25.0, snr=0.1, tilt_jitter=2.0,
              seed=0, kv=300.0, cs=2.7, ac=0.1):
    """Small RELION-style dataset with a known 1-D motion, built from the RyR1 map shipped with ManifoldEM.

    The map's periphery (the cytoplasmic 'arms') moves along z by amp_angst * tau, tau ~ U[0, 1]. Particles
    are side views on one great circle, projected (Fourier slice, RELION Euler convention), multiplied by a
    RELION-convention CTF and buried in Gaussian noise (SNR = var(signal) / var(noise))."""
    import mrcfile
    import pandas as pd
    from scipy import ndimage
    t0 = time.time()
    rng = np.random.default_rng(seed)
    os.makedirs(out, exist_ok=True)
    with mrcfile.open(src, permissive=True) as m:
        vol0, px0 = m.data.astype(np.float32), float(m.voxel_size.x)
    N0 = vol0.shape[0]
    px = px0 * N0 / box
    F = np.fft.fftshift(np.fft.fftn(vol0))
    s = N0 // 2 - box // 2
    vol = (np.fft.ifftn(np.fft.ifftshift(F[s:s + box, s:s + box, s:s + box])).real * (box / N0) ** 3).astype(np.float32)
    z, y, x = np.mgrid[:box, :box, :box].astype(np.float32)
    r = np.hypot(x - box // 2, y - box // 2) / (box / 2)
    t = np.clip((r - 0.40) / 0.30, 0, 1)
    w = t * t * (3 - 2 * t)  # 0 in the core, 1 in the periphery
    taus = np.linspace(0, 1, levels)
    amp = amp_angst / px
    vols = np.stack([ndimage.map_coordinates(vol, [z - amp * tk * w, y, x], order=3, mode='nearest') for tk in taus])
    for name, data in (('avg_volume.mrc', vols[levels // 2]), ('conformations.mrcs', vols)):
        with mrcfile.new(os.path.join(out, name), overwrite=True) as m:
            m.set_data(data.astype(np.float32))
            m.voxel_size = px
    rot, psi = rng.uniform(0, 360, n_particles), rng.uniform(0, 360, n_particles)
    tilt = 90 + rng.normal(0, tilt_jitter, n_particles)
    level = rng.integers(0, levels, n_particles)
    dU = rng.uniform(10000.0, 30000.0, n_particles)
    dV = dU + rng.normal(0, 100, n_particles)
    dang = rng.uniform(0, 180, n_particles)
    A = _relion_matrix(rot, tilt, psi)
    lam = 12.2643247 / np.sqrt(kv * 1e3 * (1 + kv * 1e3 * 0.978466e-6))
    f = np.fft.fftfreq(box, d=px)
    FX, FY = np.meshgrid(f, f)
    s2, ang = FX ** 2 + FY ** 2, np.arctan2(FY, FX)
    M = 2 * box
    k = np.arange(M) - M // 2
    KX, KY = np.meshgrid(k, k)
    clean = np.empty((n_particles, box, box), np.float32)
    for lev in range(levels):
        vp = np.zeros((M, M, M), np.float32)
        vp[M // 4:M // 4 + box, M // 4:M // 4 + box, M // 4:M // 4 + box] = vols[lev]
        F3 = np.fft.fftshift(np.fft.fftn(np.fft.ifftshift(vp))).astype(np.complex64)
        idx = np.nonzero(level == lev)[0]
        for c0 in range(0, len(idx), 250):
            j = idx[c0:c0 + 250]
            k3 = A[j, 0, :, None] * KX.ravel() + A[j, 1, :, None] * KY.ravel()  # A^T (kx, ky, 0)
            crd = np.stack([k3[:, 2], k3[:, 1], k3[:, 0]]).reshape(3, -1) + M // 2
            F2 = (ndimage.map_coordinates(F3.real, crd, order=1)
                  + 1j * ndimage.map_coordinates(F3.imag, crd, order=1)).reshape(len(j), M, M)
            img = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(F2, axes=(1, 2))), axes=(1, 2)).real
            proj = img[:, M // 4:M // 4 + box, M // 4:M // 4 + box]
            df = (0.5 * (dU[j] + dV[j]))[:, None, None] + (0.5 * (dU[j] - dV[j]))[:, None, None] * np.cos(
                2 * (ang - np.deg2rad(dang[j])[:, None, None]))
            gamma = np.pi * lam * df * s2 - 0.5 * np.pi * cs * 1e7 * lam ** 3 * s2 ** 2
            clean[j] = np.fft.ifft2(np.fft.fft2(proj) * np.sin(gamma + np.arcsin(ac))).real
        print(f'\r  projected conformation {lev + 1}/{levels}', end='', flush=True)
    print()
    sigma = np.sqrt(clean.var() / snr)
    imgs = (clean + rng.normal(0, sigma, clean.shape).astype(np.float32)) / sigma
    with mrcfile.new(os.path.join(out, 'particles.mrcs'), overwrite=True) as m:
        m.set_data(imgs.astype(np.float32))
        m.set_image_stack()
        m.voxel_size = px
    names = [f'{i + 1:06d}@particles.mrcs' for i in range(n_particles)]
    df = pd.DataFrame(dict(rlnImageName=names, rlnAngleRot=rot, rlnAngleTilt=tilt, rlnAnglePsi=psi,
                           rlnOriginXAngst=0.0, rlnOriginYAngst=0.0, rlnDefocusU=dU, rlnDefocusV=dV,
                           rlnDefocusAngle=dang, rlnOpticsGroup=1))
    with open(os.path.join(out, 'particles.star'), 'w') as fh:  # header lines must be contiguous for ManifoldEM
        fh.write('\n# version 30001\n\ndata_optics\n\nloop_\n')
        for i, c in enumerate(['rlnOpticsGroupName', 'rlnOpticsGroup', 'rlnVoltage', 'rlnSphericalAberration',
                               'rlnAmplitudeContrast', 'rlnImagePixelSize', 'rlnImageSize', 'rlnImageDimensionality']):
            fh.write(f'_{c} #{i + 1}\n')
        fh.write(f'opticsGroup1 1 {kv:.6f} {cs:.6f} {ac:.6f} {px:.6f} {box} 2\n\n\n# version 30001\n\n'
                 'data_particles\n\nloop_\n')
        fh.writelines(f'_{c} #{i + 1}\n' for i, c in enumerate(df.columns))
        df.to_csv(fh, sep=' ', header=False, index=False, float_format='%.6f')
    pd.DataFrame(dict(image_index=np.arange(n_particles), tau=taus[level], level=level)).to_csv(
        os.path.join(out, 'ground_truth.csv'), index=False)
    return dict(out=os.path.abspath(out), pixel_size=px, box=box, n_particles=n_particles,
                seconds=round(time.time() - t0, 1))


@command
def ground_truth(toml, gt_csv):
    """Per-PD Spearman correlation between ManifoldEM's final tau and the synthetic ground truth."""
    import pandas as pd
    from scipy.stats import spearmanr
    from ManifoldEM import myio
    params = _load(toml)
    prds = _prds()
    gt = pd.read_csv(gt_csv)['tau'].values
    v = myio.fin1(f'{params.traj_file}name{params.traj_name}_vars.pkl')
    rows = []
    for x in sorted(v['xSelect']):
        tau = np.asarray(v['trajTaus'][x]).ravel()
        pos_path, pos_psi1 = np.asarray(v['posPathAll'][x]), np.asarray(v['posPsi1All'][x])
        idx = np.asarray(prds.thresholded_image_indices[x])[pos_path[pos_psi1]]
        con_order = len(pos_path) // params.con_order_range  # same mapping as writeRelionS2
        raw = idx[con_order - 1:con_order - 1 + len(tau)]
        rows.append(dict(pd=int(x) + 1, rho=float(spearmanr(tau, gt[raw])[0]), n=int(len(tau))))
    return dict(rows=rows)


# ------------------------------------------------------------------------------ 3D reconstruction
def _backproject(images, rot, tilt, psi, pad=2, floor_frac=0.1, batch=32):
    """Direct Fourier inversion (trilinear splatting, RELION Euler convention, no CTF), like
    `relion_reconstruct --i x.star` on ManifoldEM's trajectory stacks (which carry no CTF or shifts)."""
    images = np.asarray(images, dtype=np.float32)
    n, N, _ = images.shape
    M = int(round(pad * N)) // 2 * 2
    Mh, off = M // 2 + 1, M // 2 - N // 2
    A = _relion_matrix(rot, tilt, psi)
    KY, KX = np.meshgrid(np.fft.fftfreq(M) * M, np.arange(Mh, dtype=float), indexing='ij')
    rmax = M / 2.0 - 1.0
    sel = KX ** 2 + KY ** 2 <= rmax ** 2
    kx, ky = KX[sel], KY[sel]
    cw = np.where(kx == 0, 0.5, 1.0)  # the kx = 0 column holds both Friedel mates
    D_re, D_im, W = (np.zeros(M * M * Mh) for _ in range(3))
    pimg = np.zeros((batch, M, M), np.float32)
    for b0 in range(0, n, batch):
        B = min(n, b0 + batch) - b0
        pimg[:] = 0
        pimg[:B, off:off + N, off:off + N] = images[b0:b0 + B]
        F = np.fft.rfft2(np.fft.ifftshift(pimg[:B], axes=(-2, -1)))[:, sel]
        k3 = kx[None, :, None] * A[b0:b0 + B, None, 0, :] + ky[None, :, None] * A[b0:b0 + B, None, 1, :]
        neg = k3[..., 0] < 0  # keep the x >= 0 half via Friedel symmetry
        k3[neg] *= -1
        F = np.where(neg, np.conj(F), F)
        x, y, z = (k3[..., i].ravel() for i in range(3))
        x0, y0, z0 = np.floor(x), np.floor(y), np.floor(z)
        fx, fy, fz = x - x0, y - y0, z - z0
        x0, y0, z0 = x0.astype(np.int64), y0.astype(np.int64), z0.astype(np.int64)
        Fr, Fi, wt = F.real.ravel(), F.imag.ravel(), np.broadcast_to(cw, F.shape).ravel()
        for dz in (0, 1):
            for dy in (0, 1):
                for dx in (0, 1):
                    w = (fx if dx else 1 - fx) * (fy if dy else 1 - fy) * (fz if dz else 1 - fz)
                    idx = (((z0 + dz) % M) * M + (y0 + dy) % M) * Mh + x0 + dx
                    D_re += np.bincount(idx, w * Fr, minlength=D_re.size)
                    D_im += np.bincount(idx, w * Fi, minlength=D_im.size)
                    W += np.bincount(idx, w * wt, minlength=W.size)
    D = (D_re + 1j * D_im).reshape(M, M, Mh)
    W = W.reshape(M, M, Mh)
    del D_re, D_im
    flip = (-np.arange(M)) % M  # x = 0 plane: add the Friedel mates
    D0, W0 = D[:, :, 0].copy(), W[:, :, 0].copy()
    D[:, :, 0] += np.conj(D0[np.ix_(flip, flip)])
    W[:, :, 0] += W0[np.ix_(flip, flip)]
    kk = (np.fft.fftfreq(M) * M).astype(np.float32)
    R = np.sqrt(kk[:, None, None] ** 2 + kk[None, :, None] ** 2 + np.arange(Mh, dtype=np.float32)[None, None, :] ** 2)
    inside = R <= rmax
    shell = np.minimum(np.rint(R), M).astype(np.int32)
    del R
    wmean = np.bincount(shell[inside], W[inside], minlength=M + 1) / np.maximum(
        np.bincount(shell[inside], minlength=M + 1), 1)
    W = np.maximum(W, floor_frac * wmean[shell])  # shell-adaptive floor stabilises gaps in Fourier space
    inside &= W > 0
    W[~inside] = 1.0
    D /= W
    D[~inside] = 0
    vol = np.fft.irfftn(D, s=(M, M, M), axes=(0, 1, 2))
    ii = np.arange(-(N // 2), N - N // 2) % M
    return vol[np.ix_(ii, ii, ii)].astype(np.float32)


def _reconstruct_one(job):
    import mrcfile
    from ManifoldEM.star import parse_star
    star, out, pad, px = job
    df = parse_star(star, 0)
    stack = os.path.join(os.path.dirname(star), df.rlnImageName.values[0].split('@')[1])
    imgs = mrcfile.read(stack).astype(np.float32)
    vol = _backproject(imgs, df.rlnAngleRot.values, df.rlnAngleTilt.values, df.rlnAnglePsi.values, pad=pad)
    with mrcfile.new(out, overwrite=True) as m:
        m.set_data(vol)
        m.voxel_size = px
    return os.path.basename(out), len(imgs)


@command
def reconstruct(toml, ncpu=1, pad=2):
    """NumPy stand-in for `manifold-cli utility mrcs2mrc` (which needs RELION's relion_reconstruct).
    Writes the same postproc/vols/EulerAngles_*.mrc files."""
    import multiprocessing
    params = _load(toml)
    os.makedirs(params.postproc_mrcs2mrc_dir, exist_ok=True)
    stars = sorted(glob.glob(os.path.join(params.bin_dir, 'EulerAngles_*.star')),
                   key=lambda f: int(re.findall(r'_(\d+)_of_', f)[-1]))
    if not stars:
        raise UserError('no trajectory STAR files found: run step 7.3 first')
    jobs = [(s, os.path.join(os.path.abspath(params.postproc_mrcs2mrc_dir), os.path.basename(s)[:-5] + '.mrc'),
             pad, params.ms_pixel_size) for s in stars]
    t0 = time.time()
    with multiprocessing.get_context('fork').Pool(max(1, int(ncpu))) as pool:
        for i, (name, n) in enumerate(pool.imap(_reconstruct_one, jobs)):
            print(f'\r  {i + 1}/{len(jobs)} volumes  ({name}, {n} images, {time.time() - t0:.0f}s)', end='', flush=True)
    print()
    return dict(n=len(jobs), seconds=round(time.time() - t0, 1), out_dir=os.path.abspath(params.postproc_mrcs2mrc_dir))


@command
def relion_selftest(exe='relion_reconstruct'):
    """Reconstruct a tiny phantom from a STAR file in ManifoldEM's own format with relion_reconstruct."""
    import subprocess
    import tempfile
    import mrcfile
    exe = shutil.which(exe)
    if not exe:
        raise UserError('relion_reconstruct is not on PATH')
    N, n, apix = 32, 300, 2.0
    blobs = [((0, 0, 0), 3.0, 1.0), ((6, 0, 0), 2.0, 0.8), ((0, 8, 0), 2.0, 0.6), ((0, 0, -9), 1.6, 1.2)]
    z, y, x = np.mgrid[0:N, 0:N, 0:N] - N // 2
    truth = sum(a * np.exp(-((x - p[0]) ** 2 + (y - p[1]) ** 2 + (z - p[2]) ** 2) / (2 * s * s)) for p, s, a in blobs)
    rng = np.random.default_rng(0)
    rot, tilt = rng.uniform(0, 360, n), np.rad2deg(np.arccos(rng.uniform(-1, 1, n)))
    A = _relion_matrix(rot, tilt, np.zeros(n))
    yy, xx = np.mgrid[0:N, 0:N] - N // 2
    imgs = np.zeros((n, N, N), np.float32)
    for p, s, a in blobs:  # analytic projections of Gaussian blobs
        q = A @ np.asarray(p, dtype=float)
        imgs += (a * np.sqrt(2 * np.pi) * s * np.exp(-((xx[None] - q[:, 0, None, None]) ** 2
                 + (yy[None] - q[:, 1, None, None]) ** 2) / (2 * s * s))).astype(np.float32)
    d = tempfile.mkdtemp(prefix='relion_selftest_')
    with mrcfile.new(os.path.join(d, 'imgs.mrcs')) as m:
        m.set_data(imgs)
    with open(os.path.join(d, 'test.star'), 'w') as f:  # same layout as ManifoldEM.star.write_star
        f.write('\ndata_ \n \nloop_ \n \n_rlnImageName #1 \n_rlnAnglePsi #2 \n_rlnAngleTilt #3 \n'
                '_rlnAngleRot #4 \n_rlnDetectorPixelSize #5 \n_rlnMagnification #6 \n')
        for i in range(n):
            f.write(f'{i + 1}@imgs.mrcs 0.0 {tilt[i]} {rot[i]} {apix} 10000.0\n')
    r = subprocess.run([exe, '--i', 'test.star', '--o', 'test.mrc'], cwd=d, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError((r.stdout + r.stderr)[-2000:])
    with mrcfile.open(os.path.join(d, 'test.mrc')) as m:
        vol, vs = m.data.astype(float), float(m.voxel_size.x)
    shutil.rmtree(d)
    cc = float(np.corrcoef(vol.ravel(), truth.ravel())[0, 1])
    return dict(exe=exe, cc=cc, pixel_size=vs, ok=bool(cc > 0.98 and abs(vs - apix) < 1e-3))


# -------------------------------------------------------------------------------- free Python code
@command
def pyexec(toml, code, fig_dir):
    """Run user code with ManifoldEM's API preloaded (used by the %%manifoldem cell magic)."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from ManifoldEM.data_store import data_store, Anchor, Sense
    params = _load(toml) if toml else None
    namespace = dict(np=np, plt=plt, params=params, data_store=data_store, Anchor=Anchor, Sense=Sense)
    if params is not None:
        namespace['prds'] = _prds()
    exec(compile(code, '<%%manifoldem>', 'exec'), namespace)
    os.makedirs(fig_dir, exist_ok=True)
    figs = []
    for num in plt.get_fignums():
        path = os.path.join(fig_dir, f'figure_{num}.png')
        plt.figure(num).savefig(path, dpi=100, bbox_inches='tight')
        figs.append(path)
    plt.close('all')
    return dict(figures=figs)


if __name__ == '__main__':
    cmd, args_file, result_file = sys.argv[1], os.path.abspath(sys.argv[2]), os.path.abspath(sys.argv[3])
    with open(args_file) as f:
        kwargs = json.load(f)
    try:
        result = COMMANDS[cmd](**kwargs)
    except UserError as exc:
        print(f'\nERROR: {exc}', flush=True)
        sys.exit(2)
    with open(result_file, 'w') as f:
        json.dump(result, f, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
'''
BRIDGE_PATH = ROOT / 'mem_bridge.py'
BRIDGE_PATH.write_text(BRIDGE_SOURCE)
print(f'✔ wrote {BRIDGE_PATH}')

In [ ]:
#@title 0.4 · Load the notebook helpers { display-mode: "form" }
#@markdown Plotting and bookkeeping functions used by the cells below. Expand the cell to read them.
# Notebook-side helpers. They run in Colab's own Python and never import ManifoldEM: pipeline steps run
# as `manifold-cli` subprocesses in the ManifoldEM environment, project queries go through mem_bridge.py,
# and plots read ManifoldEM's output files (h5 / pickle / png / gif / mrc) directly.
import base64
import glob
import io
import json
import os
import pickle
import shlex
import shutil
import signal
import subprocess
import sys
import tempfile
import time
import warnings
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Image, Markdown, display

IN_COLAB = 'google.colab' in sys.modules


def _read_pickle(path):
    with open(path, 'rb') as f, warnings.catch_warnings():
        warnings.simplefilter('ignore', DeprecationWarning)  # numpy-1 pickles opened with numpy 2
        return pickle.load(f)


def _h5(path, *keys):
    with h5py.File(path, 'r') as f:
        return {k: f[k][()] for k in (keys or f.keys()) if k in f}


def _human(nbytes):
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if nbytes < 1024 or unit == 'TB':
            return f'{nbytes:.1f} {unit}'
        nbytes /= 1024


def _du(path):
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file()) if Path(path).exists() else 0


class Session:
    """State shared by all notebook cells: paths, the current project and its latest summary."""

    def __init__(self, root, env_dir, bridge_path):
        self.root = Path(root)
        self.env = Path(env_dir)
        self.py = self.env / 'bin' / 'python'
        self.cli_exe = self.env / 'bin' / 'manifold-cli'
        self.bridge_path = Path(bridge_path)
        self.work = self.root / 'work'  # all projects live here: params_<name>.toml + output/<name>/
        self.data = self.root / 'data'
        self.tmp = self.root / 'tmp'
        for d in (self.work, self.data, self.tmp, self.work / '.notebook'):
            d.mkdir(parents=True, exist_ok=True)
        self.ncpu = os.cpu_count() or 1
        self.project = None
        self.S, self.A, self.P = {}, {}, {}

    # ----------------------------------------------------------------------------- processes
    def run(self, cmd, cwd=None, echo=True, check=True):
        """Run a command, streaming its output live; Ctrl-C / the stop button kills it and its workers."""
        cmd = [str(c) for c in cmd]
        if echo:
            shown = ' '.join(shlex.quote(c) for c in cmd).replace(str(self.env / 'bin') + '/', '')
            print(f'$ {shown}', flush=True)
        env = dict(os.environ, PYTHONUNBUFFERED='1', MPLBACKEND='Agg', OMP_NUM_THREADS='1')
        t0 = time.time()
        proc = subprocess.Popen(cmd, cwd=cwd or self.work, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, start_new_session=True)
        tail = ''
        try:
            while True:
                chunk = proc.stdout.read1(65536)
                if not chunk:
                    break
                text = chunk.decode('utf-8', 'replace')
                tail = (tail + text)[-3000:]
                sys.stdout.write(text)
                sys.stdout.flush()
            rc = proc.wait()
        except KeyboardInterrupt:
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait()
            raise
        if check and rc != 0:
            last = [l for l in tail.replace('\r', '\n').splitlines() if l.strip()]
            raise RuntimeError(f'command failed with exit code {rc}: {last[-1] if last else ""}')
        return time.time() - t0

    def cli(self, *args, ncpu=None):
        """`manifold-cli -n <ncpu> <args...> params_<project>.toml`, run from the work folder."""
        dt = self.run([self.cli_exe, '-n', ncpu or self.ncpu, *args, self.toml])
        print(f'✔ {" ".join(map(str, args))} finished in {dt / 60:.1f} min\n')
        return dt

    def bridge(self, command, **kwargs):
        """Call a mem_bridge.py command inside the ManifoldEM environment and return its JSON result."""
        d = Path(tempfile.mkdtemp(dir=self.tmp))
        try:
            (d / 'args.json').write_text(json.dumps(kwargs, default=str))
            self.run([self.py, self.bridge_path, command, d / 'args.json', d / 'result.json'], echo=False)
            return json.loads((d / 'result.json').read_text())
        finally:
            shutil.rmtree(d, ignore_errors=True)

    # ------------------------------------------------------------------------------- projects
    @property
    def toml(self):
        if not self.project:
            raise RuntimeError('No project selected: run step 2.1 (create) or 2.2 (re-open) first.')
        return str(self.work / f'params_{self.project}.toml')

    def projects(self):
        return sorted(p.name[len('params_'):-len('.toml')] for p in self.work.glob('params_*.toml'))

    def use(self, name):
        if not (self.work / f'params_{name}.toml').exists():
            raise FileNotFoundError(f'no project called {name!r} in {self.work} (found: {self.projects()})')
        self.project = name
        return self.refresh()

    @property
    def state_file(self):
        return self.work / '.notebook' / f'{self.project}.json'

    def state(self):
        return json.loads(self.state_file.read_text()) if self.state_file.exists() else {}

    def save_state(self, **values):
        st = self.state()
        st.update(values)
        self.state_file.write_text(json.dumps(st, indent=1, default=str))

    def refresh(self):
        """Re-read the project (after every step, since subprocesses change it on disk)."""
        npz = self.tmp / f'summary_{self.project}.npz'
        self.S = self.bridge('summary', toml=self.toml, out_npz=str(npz))
        with np.load(npz) as z:
            self.A = {k: z[k] for k in z.files}
        self.P = self.S['paths']
        return self.S

    def status(self):
        d = self.S.get('done', {})
        steps = [('binning', 'Bin particles into PDs (3)'), ('calc_distance', 'Distances (4)'),
                 ('manifold_analysis', 'Diffusion-map embedding (4)'), ('psi_analysis', 'NLSA (4)'),
                 ('nlsa_movie', 'Movies (4)'), ('find_ccs', 'Conformational coordinates (7.1)'),
                 ('probability_landscape', 'Energy landscape (7.2)'), ('trajectory', '3D trajectories (7.3)')]
        rows = ''.join(f"<tr><td>{'✅' if d.get(k) else '⬜'}</td><td>{label}</td></tr>" for k, label in steps)
        nv = d.get('volumes', 0)
        rows += f"<tr><td>{'✅' if nv else '⬜'}</td><td>Volumes (8){f' — {nv}' if nv else ''}</td></tr>"
        display(HTML(f"<b>Project <code>{self.project}</code></b> — {self.S.get('n_pd', 0)} PDs, "
                     f"ManifoldEM {self.S.get('version')}<table>{rows}</table>"))

    # ------------------------------------------------------------------------------ geometry
    def _disk(self, vecs):
        """Lambert azimuthal equal-area map of unit vectors around the hemisphere axis (edge = 90°)."""
        axis = np.asarray(self.S.get('hemisphere_vec', [1.0, 0.0, 0.0]), dtype=float)
        axis /= np.linalg.norm(axis)
        ref = np.array([0.0, 0.0, 1.0]) if abs(axis[2]) < 0.9 else np.array([0.0, 1.0, 0.0])
        e1 = np.cross(ref, axis)
        e1 /= np.linalg.norm(e1)
        e2 = np.cross(axis, e1)
        v = np.asarray(vecs, dtype=float)
        c = np.clip(axis @ v, -1, 1)
        r = np.sqrt(np.maximum(2 * (1 - c), 0))
        az = np.arctan2(e2 @ v, e1 @ v)
        return r * np.cos(az), r * np.sin(az)

    def _disk_axes(self, ax, title=''):
        for deg in (30, 60, 90):
            r = np.sqrt(2 * (1 - np.cos(np.deg2rad(deg))))
            ax.add_patch(plt.Circle((0, 0), r, fill=False, lw=0.6, color='0.75', ls='-' if deg == 90 else ':'))
            ax.text(r * 0.71, r * 0.71, f'{deg}°', fontsize=7, color='0.55')
        ax.set_aspect('equal')
        ax.set_xlim(-1.5, 1.5)
        ax.set_ylim(-1.5, 1.5)
        ax.axis('off')
        ax.set_title(title, fontsize=10)

    # ---------------------------------------------------------------------------- 1. inputs
    @staticmethod
    def show_inputs(report, npz=None):
        ok = not report.get('problems')
        lines = [f"**{'✅ Inputs look usable' if ok else '❌ ManifoldEM will not be able to use these inputs'}**", '']
        if 'n_particles' in report:
            lines.append(f"- {report['n_particles']:,} particles in the STAR file, {report['n_images']:,} images of "
                         f"{report['box']}×{report['box']} px in the stack")
        px = [f'{v:.4g} Å ({k})' for k, v in (('STAR', report.get('star_pixel_size')),
                                               ('MRC header', report.get('header_pixel_size'))) if v]
        if px:
            lines.append('- pixel size: ' + ', '.join(px))
        if 'voltage' in report:
            lines.append(f"- {report['voltage']:g} kV, Cs {report.get('cs', float('nan')):g} mm, "
                         f"amplitude contrast {report.get('amp_contrast', float('nan')):g}")
        lines += [f'- ❌ {p}' for p in report.get('problems', [])] + [f'- ⚠️ {w}' for w in report.get('warnings', [])]
        display(Markdown('\n'.join(lines)))
        if not npz or not os.path.exists(npz):
            return
        with np.load(npz) as z:
            imgs, idx = z['images'], z['image_index']
            rot, tilt, defocus = z['rot'], z['tilt'], z['defocus']
        n = len(imgs)
        fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(1.6 * ((n + 1) // 2), 3.4))
        for ax in axes.ravel():
            ax.axis('off')
        for ax, im, i in zip(axes.ravel(), imgs, idx):
            lo, hi = np.percentile(im, [1, 99])
            ax.imshow(im, cmap='gray', vmin=lo, vmax=hi)
            ax.set_title(f'#{i + 1}', fontsize=8)
        fig.suptitle('Random particle images', fontsize=10)
        plt.tight_layout()
        plt.show()
        fig = plt.figure(figsize=(12, 3.6))
        ax = fig.add_subplot(1, 2, 1, projection='mollweide')
        lon = np.deg2rad((rot + 180) % 360 - 180)
        lat = np.deg2rad(90 - tilt)
        ax.scatter(lon, lat, s=1, alpha=0.25, color='tab:blue', rasterized=True)
        ax.grid(True, lw=0.4)
        ax.set_title('Viewing directions (rot, tilt)', fontsize=10)
        ax.tick_params(labelsize=7)
        ax2 = fig.add_subplot(1, 2, 2)
        ax2.hist(defocus / 1e4, bins=60, color='tab:gray')
        ax2.set_xlabel('mean defocus (µm)')
        ax2.set_ylabel('particles')
        ax2.set_title('Defocus distribution', fontsize=10)
        plt.tight_layout()
        plt.show()

    # --------------------------------------------------------------------------- 3. binning
    def show_binning(self):
        S, A = self.S, self.A
        occ_full, centres = A['occupancy_full'], A['bin_centers']
        low, high = S['thres_low'], S['thres_high']
        fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw=dict(width_ratios=[1.1, 1]))
        x, y = self._disk(centres)
        nz = occ_full > 0
        sc = ax.scatter(x[nz], y[nz], c=np.log10(occ_full[nz]), s=14, cmap='viridis', vmin=0)
        keep = occ_full >= low
        ax.scatter(x[keep], y[keep], s=60, facecolors='none', edgecolors='crimson', lw=1.2,
                   label=f'PD (≥ {low} particles)')
        cb = plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
        cb.set_label('log10(particles per bin)')
        ax.legend(loc='lower left', fontsize=8, frameon=False)
        self._disk_axes(ax, 'Orientation bins on the half-sphere')
        ax2.hist(occ_full[nz], bins=min(80, max(10, nz.sum() // 3)), color='tab:gray')
        ax2.axvline(low, color='crimson', lw=1.5, label=f'low threshold = {low}')
        ax2.axvline(high, color='tab:green', lw=1.5, ls='--', label=f'high threshold (cap) = {high}')
        ax2.set_xlabel('particles per orientation bin')
        ax2.set_ylabel('bins')
        ax2.legend(fontsize=8)
        ax2.set_title('Bin occupancy', fontsize=10)
        plt.tight_layout()
        plt.show()
        msg = (f"**{S['n_pd']} projection directions (PDs)** out of {S['n_bins']} bins · "
               f"{S['n_used']:,} of {S['n_particles']:,} particles used · {S['n_clusters']} connected "
               f"component(s) · Shannon angle {S['shannon_angle']:.4f} rad · bin width {S['ang_width']:.4f} rad "
               f"(aperture index {S['aperture_index']})")
        warn = []
        if S['n_pd'] < 3:
            warn.append('fewer than 3 PDs: lower the low threshold or raise the aperture index')
        if low < 2 * S['con_order_range']:
            warn.append(f"the low threshold ({low}) is below 2 × con_order_range ({2 * S['con_order_range']}): "
                        'NLSA needs more images per PD; raise it or lower con_order_range in step 4.0')
        if S['n_clusters'] > 1:
            warn.append(f"{S['n_clusters']} separate groups of PDs: each group needs its own anchor in step 6")
        display(Markdown(msg + ''.join(f'\n- ⚠️ {w}' for w in warn)))

    def show_sphere_3d(self):
        import plotly.graph_objects as go
        A = self.A
        u, v = np.mgrid[0:2 * np.pi:40j, 0:np.pi:20j]
        sphere = go.Surface(x=np.cos(u) * np.sin(v), y=np.sin(u) * np.sin(v), z=np.cos(v), opacity=0.15,
                            showscale=False, colorscale=[[0, 'lightgrey'], [1, 'lightgrey']], hoverinfo='skip')
        p = A['pos_pd']
        pds = go.Scatter3d(x=p[0], y=p[1], z=p[2], mode='markers+text', text=[str(i + 1) for i in range(p.shape[1])],
                           textfont=dict(size=9), marker=dict(size=6, color=A['cluster_ids'], colorscale='Turbo'),
                           hovertext=[f'PD {i + 1}: {o} particles, cluster {c}' for i, (o, c) in
                                      enumerate(zip(A['occupancy'], A['cluster_ids']))], hoverinfo='text', name='PDs')
        q = A['pos_particles']
        parts = go.Scatter3d(x=q[0], y=q[1], z=q[2], mode='markers', marker=dict(size=1.5, color='grey', opacity=0.3),
                             hoverinfo='skip', name='particles (sample)')
        fig = go.Figure([sphere, parts, pds])
        fig.update_layout(height=550, margin=dict(l=0, r=0, t=30, b=0), title='PDs on the sphere (hover for details)',
                          scene=dict(aspectmode='cube', xaxis_visible=False, yaxis_visible=False, zaxis_visible=False))
        fig.show()

    # ----------------------------------------------------------------------------- 5. PDs
    def pd_table(self, top=None):
        import pandas as pd
        A, S = self.A, self.S
        anchors = {int(a[0]): a for a in A['anchors']}
        trash = set(int(t) for t in A['trash'])
        role = []
        for i in range(S['n_pd']):
            if i in anchors:
                role.append(f"anchor ψ{anchors[i][1]} {'FWD' if anchors[i][2] > 0 else 'REV'}")
            else:
                role.append('removed' if i in trash else '')
        df = pd.DataFrame(dict(PD=np.arange(1, S['n_pd'] + 1), particles=A['occupancy'], used=A['n_used'],
                               cluster=A['cluster_ids'], theta=np.round(A['theta'], 1), phi=np.round(A['phi'], 1),
                               role=role))
        df = df.sort_values('particles', ascending=False)
        return df.head(top) if top else df

    def _gif_html(self, path, width=150, label=''):
        """Re-encode ManifoldEM's GIF (which plays once) as a looping GIF, downscaled for display."""
        from PIL import Image as PILImage, ImageSequence
        if not os.path.exists(path):
            return f'<div style="width:{width}px;font-size:11px;color:#999">{label}<br>(missing)</div>'
        with PILImage.open(path) as im:
            frames = [f.convert('L') for f in ImageSequence.Iterator(im)]
            duration = im.info.get('duration', 200)
        if frames[0].width > width:
            frames = [f.resize((width, width), PILImage.LANCZOS) for f in frames]
        buf = io.BytesIO()
        frames[0].save(buf, format='GIF', save_all=True, append_images=frames[1:], loop=0, duration=duration)
        b64 = base64.b64encode(buf.getvalue()).decode()
        return (f'<div style="display:inline-block;margin:2px;text-align:center;font-size:11px">'
                f'<img src="data:image/gif;base64,{b64}" width="{width}"><br>{label}</div>')

    def movies_html(self, pd1, psis=None, width=150):
        psis = psis or range(1, self.S['num_psi'] + 1)
        return ''.join(self._gif_html(self.P['gif'].format(p=pd1, k=k), width, f'ψ{k}') for k in psis)

    def _pd_header(self, pd1):
        A = self.A
        i = pd1 - 1
        anchors = {int(a[0]): a for a in A['anchors']}
        role = ('anchor ψ%d %s' % (anchors[i][1], 'FWD' if anchors[i][2] > 0 else 'REV')) if i in anchors else (
            'removed' if i in set(A['trash'].tolist()) else '')
        return (f"<h4 style='margin:4px 0'>PD {pd1} · {A['occupancy'][i]} particles ({A['n_used'][i]} used) · "
                f"cluster {A['cluster_ids'][i]} · θ={A['theta'][i]:.1f}° φ={A['phi'][i]:.1f}°"
                f"{' · <b>' + role + '</b>' if role else ''}</h4>")

    def _pd_figure(self, pd1):
        """Class average, eigenvalue spectrum, kernel-bandwidth fit and diffusion-map embedding of one PD."""
        S, P = self.S, self.P
        i = pd1 - 1
        fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
        ca = P['class_avg'].format(p=pd1)
        if os.path.exists(ca):
            axes[0].imshow(plt.imread(ca).T, cmap='gray')  # transposed to match the movie frames
        axes[0].set_title('class average', fontsize=9)
        axes[0].axis('off')
        es = P['eig_spec'].format(p=pd1)
        if os.path.exists(es):
            e = np.atleast_2d(np.loadtxt(es))
            axes[1].bar(e[:, 0], e[:, 1], color=['tab:blue' if j <= S['num_psi'] else 'lightgrey' for j in e[:, 0]])
            axes[1].set_xlabel('eigenvector')
        axes[1].set_title(f'eigenvalues (blue: ψ1–ψ{S["num_psi"]} analysed)', fontsize=9)
        axes[2].axis('off')
        axes[3].axis('off')
        psi_file = P['psi'].format(p=i)
        if os.path.exists(psi_file):
            d = _h5(psi_file)
            if {'logEps', 'logSumWij', 'popt'} <= set(d):  # absent after a manual re-embedding
                popt = d['popt']
                axes[2].axis('on')
                axes[2].plot(d['logEps'], d['logSumWij'], '.', ms=2, color='tab:gray')
                axes[2].plot(d['logEps'], popt[3] + popt[2] * np.tanh(popt[0] * d['logEps'] + popt[1]), 'tab:red')
                axes[2].axvline(-popt[1] / popt[0], color='tab:orange', ls='--', lw=1)
                axes[2].set_xlabel('ln ε')
                axes[2].set_ylabel('ln Σ W')
                axes[2].set_title(f'kernel bandwidth (dim ≈ {2 * popt[0] * popt[2]:.1f})', fontsize=9)
            psi = d['psi']
            rot = None
            dist_file = P['dist'].format(p=i)
            if os.path.exists(dist_file) and 'posPath' in d:
                rot = _h5(dist_file, 'rotations').get('rotations')
                rot = None if rot is None else rot[np.asarray(d['posPath']).ravel()]
            axes[3].axis('on')
            s = axes[3].scatter(psi[:, 0], psi[:, 1], c=rot, s=6, cmap='twilight' if rot is not None else None)
            if rot is not None:
                plt.colorbar(s, ax=axes[3], fraction=0.05, pad=0.02).set_label('in-plane rotation (°)', fontsize=8)
            axes[3].set_xlabel('ψ1')
            axes[3].set_ylabel('ψ2')
            axes[3].set_title('diffusion map embedding', fontsize=9)
        plt.tight_layout()
        return fig

    def _pd_movies(self, pd1, width=150):
        return f'<div><b>NLSA movies</b> (one per eigenvector ψ)</div>{self.movies_html(pd1, width=width)}'

    def show_pd(self, pd1, diagnostics=True, width=150):
        """Everything ManifoldEM computed for one PD (1-based, as in topos/PrD_<n>)."""
        if not 1 <= pd1 <= self.S['n_pd']:
            print(f'PD must be between 1 and {self.S["n_pd"]}')
            return
        display(HTML(self._pd_header(pd1)))
        if diagnostics:
            self._pd_figure(pd1)
            plt.show()
        display(HTML(self._pd_movies(pd1, width)))

    def pd_browser(self, width=150):
        """Dropdown + slider over PDs. Built from HTML/Image widgets (not an Output widget), which update by value."""
        import ipywidgets as w
        order = self.pd_table()
        options = [(f"PD {r.PD} · {r.particles} particles · cluster {r.cluster}{' · ' + r.role if r.role else ''}",
                    int(r.PD)) for r in order.itertuples()]
        dropdown = w.Dropdown(options=options, description='Jump to', layout=w.Layout(width='520px'))
        slider = w.IntSlider(value=options[0][1], min=1, max=self.S['n_pd'], description='PD',
                             continuous_update=False, layout=w.Layout(width='520px'))
        header, movies = w.HTML(), w.HTML()
        figure = w.Image(format='png', layout=w.Layout(max_width='100%'))

        def render(_=None):
            header.value = self._pd_header(slider.value)
            fig = self._pd_figure(slider.value)
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=80)
            plt.close(fig)
            figure.value = buf.getvalue()
            movies.value = self._pd_movies(slider.value, width)

        def jump(change):
            slider.value = change['new']

        dropdown.observe(jump, names='value')
        slider.observe(render, names='value')
        render()
        display(w.VBox([dropdown, slider, header, figure, movies]))

    # ----------------------------------------------------------------------------- 6. anchors
    def anchor_candidates(self, per_cluster=2, psis=(1, 2, 3), width=120):
        df = self.pd_table()
        html = ['<table style="border-collapse:collapse">']
        for cluster in sorted(df.cluster.unique()):
            sub = df[df.cluster == cluster].head(per_cluster)
            html.append(f'<tr><td colspan="{len(psis) + 1}" style="padding-top:8px"><b>Cluster {cluster}</b> '
                        f'({(df.cluster == cluster).sum()} PDs) — most populated PDs:</td></tr>')
            for r in sub.itertuples():
                cells = ''.join(f'<td>{self._gif_html(self.P["gif"].format(p=r.PD, k=k), width, f"PD {r.PD} · ψ{k}")}</td>'
                                for k in psis)
                html.append(f'<tr><td style="padding-right:8px;font-size:12px">PD <b>{r.PD}</b><br>{r.particles} '
                            f'particles</td>{cells}</tr>')
        display(HTML(''.join(html) + '</table>'))

    def parse_anchors(self, text):
        """'auto' or 'PD:psi:FWD|REV, ...' (1-based PD and ψ) -> [[pd0, psi, sense], ...]."""
        text = text.strip()
        if text.lower() == 'auto':
            df = self.pd_table()
            best = df.groupby('cluster', sort=True).head(1)
            if df.cluster.nunique() > 1:
                display(Markdown('⚠️ **auto** picked one anchor per cluster, all `FWD`. Watch their ψ1 movies '
                                 'above: if the motion runs the opposite way in some cluster, mark that anchor `REV`.'))
            return [[int(r.PD) - 1, 1, 'FWD'] for r in best.itertuples()]
        out = []
        for item in filter(None, (s.strip() for s in text.replace(';', ',').split(','))):
            parts = [p.strip() for p in item.split(':')]
            if len(parts) == 2:
                parts.append('FWD')
            if len(parts) != 3:
                raise ValueError(f'cannot read anchor {item!r}: use PD:psi:FWD or PD:psi:REV, e.g. 12:1:FWD')
            out.append([int(parts[0]) - 1, int(parts[1]), parts[2].upper()])
        return out

    @staticmethod
    def parse_pd_list(text):
        out = []
        for item in filter(None, (s.strip() for s in text.replace(';', ',').split(','))):
            if '-' in item:
                a, b = item.split('-')
                out += list(range(int(a), int(b) + 1))
            else:
                out.append(int(item))
        return [p - 1 for p in out]

    def show_anchor_map(self, title='PDs, anchors and removed PDs'):
        A, S = self.A, self.S
        fig, ax = plt.subplots(figsize=(6.5, 6.5))
        x, y = self._disk(A['pos_pd'])
        for a, b in A['edges']:
            ax.plot([x[a], x[b]], [y[a], y[b]], color='0.85', lw=0.8, zorder=0)
        ax.scatter(x, y, c=A['cluster_ids'], cmap='tab10', vmin=0, vmax=9, s=40 + 160 * A['occupancy'] / A['occupancy'].max(),
                   edgecolors='k', lw=0.4, zorder=1)
        for i in range(S['n_pd']):
            ax.text(x[i], y[i], str(i + 1), fontsize=7, ha='center', va='center', zorder=3)
        for pd0, psi, sense in A['anchors']:
            ax.scatter(x[pd0], y[pd0], marker='*', s=420, facecolors='gold', edgecolors='k', zorder=2)
            ax.annotate(f"ψ{psi} {'FWD' if sense > 0 else 'REV'}", (x[pd0], y[pd0]), xytext=(8, 8),
                        textcoords='offset points', fontsize=8, weight='bold')
        for t in A['trash']:
            ax.scatter(x[t], y[t], marker='x', s=160, color='red', lw=2, zorder=4)
        self._disk_axes(ax, title + '\n(colour = connected component, ★ = anchor, ✕ = removed, size = particles)')
        plt.show()

    # ------------------------------------------------------------------------- 7. CC & landscape
    def show_cc(self):
        A, S, P = self.A, self.S, self.P
        cc = _read_pickle(P['CC_file'])
        psinums, senses = np.asarray(cc['psinums'][0]), np.asarray(cc['senses'][0])
        fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw=dict(width_ratios=[1, 1.2]))
        x, y = self._disk(A['pos_pd'])
        for a, b in A['edges']:
            ax.plot([x[a], x[b]], [y[a], y[b]], color='0.88', lw=0.8, zorder=0)
        cmap = plt.get_cmap('tab10')
        for i in range(S['n_pd']):
            if psinums[i] < 0:
                ax.scatter(x[i], y[i], marker='x', color='0.5', s=60)
                continue
            ax.scatter(x[i], y[i], marker='^' if senses[i] > 0 else 'v', s=120, color=cmap(psinums[i] % 10),
                       edgecolors='k', lw=0.4)
            ax.text(x[i], y[i] - 0.09, str(i + 1), fontsize=6, ha='center', va='top')
        self._disk_axes(ax, 'Chosen eigenvector per PD (colour), sense ▲ FWD / ▼ REV, ✕ unassigned')
        k = S['num_psi']
        counts = [(np.sum((psinums == j) & (senses > 0)), np.sum((psinums == j) & (senses < 0))) for j in range(k)]
        ax2.bar(np.arange(1, k + 1), [c[0] for c in counts], color=[cmap(j % 10) for j in range(k)], label='FWD')
        ax2.bar(np.arange(1, k + 1), [c[1] for c in counts], bottom=[c[0] for c in counts],
                color=[cmap(j % 10) for j in range(k)], alpha=0.45, hatch='//', label='REV')
        ax2.set_xticks(np.arange(1, k + 1))
        ax2.set_xticklabels([f'ψ{j}' for j in range(1, k + 1)])
        ax2.set_ylabel('PDs')
        ax2.legend()
        ax2.set_title(f'{np.sum(psinums >= 0)} of {len(psinums)} PDs assigned a conformational coordinate', fontsize=10)
        plt.tight_layout()
        plt.show()
        return psinums, senses

    def show_landscape(self, gt_rows=None):
        S, P = self.S, self.P
        hUn = np.asarray(_read_pickle(P['traj'])['hUn'], dtype=float)
        p = hUn / hUn.sum()
        with np.errstate(divide='ignore'):
            dG = -np.log(p)
        dG -= np.nanmin(dG[np.isfinite(dG)])
        states = np.arange(1, len(hUn) + 1)
        ncol = 3 if gt_rows else 2
        fig, axes = plt.subplots(1, ncol, figsize=(5.2 * ncol, 3.8))
        axes[0].bar(states, hUn, color='tab:blue', width=0.85)
        axes[0].set_xlabel('conformational state (τ bin)')
        axes[0].set_ylabel('particles')
        axes[0].set_title(f'Occupancy along the coordinate ({int(hUn.sum()):,} images)', fontsize=10)
        axes[1].plot(states, dG, 'o-', color='crimson', ms=3)
        axes[1].set_xlabel('conformational state (τ bin)')
        axes[1].set_ylabel('ΔG = −ln P  (k$_B$T)')
        axes[1].set_title('Free-energy profile', fontsize=10)
        if gt_rows:
            rho = np.array([r['rho'] for r in gt_rows])
            axes[2].bar([r['pd'] for r in gt_rows], rho, color=np.where(rho > 0, 'tab:green', 'tab:orange'))
            axes[2].axhline(0, color='k', lw=0.6)
            axes[2].set_ylim(-1, 1)
            axes[2].set_xlabel('PD')
            axes[2].set_ylabel('Spearman ρ (recovered τ vs truth)')
            axes[2].set_title(f'Synthetic ground truth: median |ρ| = {np.median(np.abs(rho)):.2f}', fontsize=10)
        plt.tight_layout()
        plt.show()
        if gt_rows:
            same = max(np.sum(rho > 0), np.sum(rho < 0))
            display(Markdown(f'{same} of {len(rho)} PDs agree on the direction of the motion '
                             '(a consistent sign means belief propagation got the senses right; the overall sign '
                             'is arbitrary).'))
        return states, hUn, dG

    # ---------------------------------------------------------------------------- 8. volumes
    @staticmethod
    def volume_files(folder):
        import re
        return sorted(glob.glob(os.path.join(folder, '*.mrc')),
                      key=lambda f: int((re.findall(r'_(\d+)_of_', f) or ['0'])[-1]))


def read_mrc(path):
    import mrcfile
    with mrcfile.open(path, permissive=True) as m:
        return np.array(m.data, dtype=np.float32, copy=True), float(m.voxel_size.x) or 1.0  # copy: read-only arrays


def needs_flip(files, mode='auto'):
    """ManifoldEM writes the trajectory images with inverted contrast, so volumes often come out negative."""
    if mode in ('yes', 'no'):
        return mode == 'yes'
    vol, _ = read_mrc(files[len(files) // 2])
    return abs(np.percentile(vol, 0.05)) > abs(np.percentile(vol, 99.95))


def volume_projections(files, flip=False):
    """(states, 3, N, N) projections along z, y and x, reading one volume at a time."""
    out = []
    for f in files:
        v, _ = read_mrc(f)
        v = -v if flip else v
        out.append(np.stack([v.sum(axis=a) for a in (0, 1, 2)]))
    return np.stack(out)


def show_volume_strip(proj, n=8, title=''):
    idx = np.unique(np.linspace(0, len(proj) - 1, min(n, len(proj))).round().astype(int))
    fig, axes = plt.subplots(3, len(idx), figsize=(1.8 * len(idx), 5.6))
    axes = np.atleast_2d(axes).reshape(3, len(idx))
    for row, name in enumerate(('projection along z', 'along y', 'along x')):
        lo, hi = np.percentile(proj[idx, row], [1, 99.8])
        for col, j in enumerate(idx):
            axes[row, col].imshow(proj[j, row], cmap='gray', vmin=lo, vmax=hi)
            axes[row, col].axis('off')
            if row == 0:
                axes[row, col].set_title(f'state {j + 1}', fontsize=8)
        axes[row, 0].text(-0.15, 0.5, name, transform=axes[row, 0].transAxes, rotation=90, va='center',
                          ha='right', fontsize=8)
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()


def volume_movie(proj, out_gif, fps=8, size=128, max_states=24):
    """Looping GIF of the three projections, played forward and back along the conformational coordinate."""
    from PIL import Image as PILImage
    limits = [np.percentile(proj[:, r], [1, 99.8]) for r in range(3)]
    frames = []
    keep = np.unique(np.linspace(0, len(proj) - 1, min(max_states, len(proj))).round().astype(int)).tolist()
    for j in keep + keep[-2:0:-1]:
        canvas = PILImage.new('L', (3 * size + 8, size), 255)
        for r, (lo, hi) in enumerate(limits):
            t = np.clip((proj[j, r] - lo) / (hi - lo + 1e-12), 0, 1)
            canvas.paste(PILImage.fromarray((255 * t).astype(np.uint8)).resize((size, size), PILImage.LANCZOS),
                         (r * (size + 4), 0))
        frames.append(canvas)
    frames[0].save(out_gif, save_all=True, append_images=frames[1:], loop=0, duration=int(1000 / fps))
    b64 = base64.b64encode(open(out_gif, 'rb').read()).decode()
    display(HTML(f'<img src="data:image/gif;base64,{b64}"><br><small>{out_gif} — projections along z, y and x, '
                 'played along the conformational coordinate and back</small>'))


def isosurface_movie(files, states, flip=False, sigma=3.0, step=2):
    """Plotly isosurfaces (marching cubes) with a slider over states; renders with plain fig.show()."""
    import plotly.graph_objects as go
    from skimage.measure import marching_cubes
    ref, apix = read_mrc(files[len(files) // 2])
    ref = -ref if flip else ref
    level = float(ref.mean() + sigma * ref.std())
    meshes = []
    for j in states:
        v, _ = read_mrc(files[j])
        v = -v if flip else v
        try:
            verts, faces, _, _ = marching_cubes(v, level=level, step_size=step, spacing=(apix,) * 3)
        except (ValueError, RuntimeError):  # no surface at this level
            verts, faces = np.zeros((3, 3)), np.array([[0, 1, 2]])
        z, y, x = verts.T
        meshes.append(go.Mesh3d(x=x, y=y, z=z, i=faces[:, 0], j=faces[:, 1], k=faces[:, 2], color='#9ab',
                                lighting=dict(ambient=0.45, diffuse=0.8, specular=0.25)))
    size = ref.shape[-1] * apix
    axis = dict(range=[0, size], visible=False)
    fig = go.Figure(data=[meshes[0]], frames=[go.Frame(data=[m], name=str(j + 1)) for m, j in zip(meshes, states)])
    fig.update_layout(
        height=600, margin=dict(l=0, r=0, t=40, b=0),
        title=f'Isosurface at mean + {sigma:g}σ — drag the slider (or press play) to move along the coordinate',
        scene=dict(aspectmode='cube', xaxis=axis, yaxis=axis, zaxis=axis),
        sliders=[dict(currentvalue=dict(prefix='state '), pad=dict(t=10), steps=[
            dict(method='animate', label=str(j + 1), args=[[str(j + 1)], dict(mode='immediate', frame=dict(duration=0))])
            for j in states])],
        updatemenus=[dict(type='buttons', x=0.05, y=0.05, buttons=[dict(label='▶ play', method='animate', args=[
            None, dict(frame=dict(duration=300, redraw=True), fromcurrent=True, mode='immediate')])])])
    fig.show()


def denoise_volumes(files, out_dir, sigma=1.0, svd_rank=0):
    """Gaussian-filter each volume and/or keep only the leading SVD components across the trajectory."""
    import mrcfile
    from scipy import ndimage
    os.makedirs(out_dir, exist_ok=True)
    stack = None
    if svd_rank:  # needs all volumes in memory at once
        vols = [read_mrc(f)[0] for f in files]
        X = np.stack(vols).reshape(len(vols), -1).astype(np.float64)
        del vols
        mean = X.mean(0)
        U, s, Vt = np.linalg.svd(X - mean, full_matrices=False)
        k = min(int(svd_rank), len(s))
        stack = (mean + (U[:, :k] * s[:k]) @ Vt[:k]).astype(np.float32)
        print(f'kept {k} SVD components ({100 * np.sum(s[:k] ** 2) / np.sum(s ** 2):.1f}% of the variance)')
        del X, U, Vt
    paths = []
    for i, f in enumerate(files):
        v, apix = read_mrc(f)
        if stack is not None:
            v = stack[i].reshape(v.shape)
        if sigma:
            v = ndimage.gaussian_filter(v, sigma).astype(np.float32)
        p = os.path.join(out_dir, os.path.basename(f))
        with mrcfile.new(p, overwrite=True) as m:
            m.set_data(v.astype(np.float32))
            m.voxel_size = apix  # keep the pixel size (ManifoldEM's own `utility denoise` drops it)
        paths.append(p)
    return paths


def register_manifoldem_magic(session):
    """%%manifoldem: run a cell inside the ManifoldEM environment with params/data_store/prds preloaded."""
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def manifoldem(line, cell):
        fig_dir = tempfile.mkdtemp(dir=session.tmp)
        res = session.bridge('pyexec', toml=session.toml if session.project else '', code=cell, fig_dir=fig_dir)
        for f in res.get('figures', []):
            display(Image(filename=f))
        shutil.rmtree(fig_dir, ignore_errors=True)


mem = Session(ROOT, ENV_DIR, BRIDGE_PATH)
register_manifoldem_magic(mem)
_dataset_file = mem.data / 'current_dataset.json'
DATA = json.loads(_dataset_file.read_text()) if _dataset_file.exists() else None
print(f'✔ helpers loaded · {mem.ncpu} CPUs will be used')
if mem.projects():
    print(f'existing projects: {", ".join(mem.projects())} — use 2.2 to re-open one')

## 1 · Data

ManifoldEM starts from a **3D refinement** of your particles (RELION, cryoSPARC → RELION STAR, ...). It needs:

* a **STAR file** with, for every particle, `rlnAngleRot`, `rlnAngleTilt`, `rlnAnglePsi`, `rlnDefocusU`,
  `rlnDefocusV` and (optionally) origins `rlnOriginX/Y[Angst]`, plus `rlnVoltage`, `rlnSphericalAberration` and
  `rlnAmplitudeContrast` (per particle or in a `data_optics` block);
* **one** `.mrcs` particle stack whose *i*-th image is the *i*-th STAR row (combine several stacks with
  `relion_stack_create`). Ideally re-extract the particles centred, with origins set to 0;
* optionally an average volume (only used for display) and a mask.

Choose a dataset in **1.1**, then run the matching cell (1.2, 1.3 or 1.4). The other two will skip themselves.

In [ ]:
#@title 1.1 · Choose a dataset { display-mode: "form" }
DATASET = "Synthetic demo (generated here, ~1 min)"  #@param ["Synthetic demo (generated here, ~1 min)", "RyR1 tutorial data (download, several GB)", "My own data (Google Drive, URL or upload)"]
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
#@markdown Tick **USE_GOOGLE_DRIVE** to mount your Drive at `/content/drive`: needed to read data from Drive, and for checkpoints (section 10).
DATASET_KIND = {'Synthetic': 'synthetic', 'RyR1 tuto': 'ryr1', 'My own da': 'own'}[DATASET[:9]]
if USE_GOOGLE_DRIVE:
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print('Google Drive can only be mounted inside Colab; skipping.')
print(f"Dataset: {DATASET} → now run cell {dict(synthetic='1.2', ryr1='1.3', own='1.4')[DATASET_KIND]}.")

In [ ]:
#@title 1.2 · Synthetic demo dataset { display-mode: "form" }
#@markdown A small dataset with a **known** motion: the RyR1 map that ships with ManifoldEM, whose four cytoplasmic
#@markdown arms move up and down by `MOTION_AMPLITUDE_A` (uniformly sampled). The particles are side views on one great
#@markdown circle, 64 px at 5.25 Å/px, with CTF and Gaussian noise. Section 7 compares ManifoldEM's answer with the truth.
N_PARTICLES = 6000  #@param {type:"slider", min:3000, max:20000, step:1000}
SNR = 0.1  #@param {type:"number"}
MOTION_AMPLITUDE_A = 25  #@param {type:"number"}
SEED = 0  #@param {type:"integer"}
import urllib.request
if DATASET_KIND != 'synthetic':
    print(f'skipped (1.1 selected: {DATASET})')
else:
    src = mem.data / 'RyR1_AvgVol_Binned.mrc'
    if not src.exists():
        base = MANIFOLDEM_REPO.strip().rstrip('/')
        urls = [base.replace('github.com', 'raw.githubusercontent.com') + f'/{MANIFOLDEM_VERSION.strip() or "main"}/demo/RyR1_AvgVol_Binned.mrc',
                'https://raw.githubusercontent.com/flatironinstitute/ManifoldEM/main/demo/RyR1_AvgVol_Binned.mrc']
        for url in urls:
            try:
                urllib.request.urlretrieve(url, src)
                break
            except Exception as exc:
                print(f'could not download {url}: {exc}')
        else:
            raise RuntimeError('could not download the RyR1 map that the synthetic data is built from')
    out = mem.data / 'synthetic'
    info = mem.bridge('synthetic', src=str(src), out=str(out), n_particles=int(N_PARTICLES), box=64,
                      amp_angst=float(MOTION_AMPLITUDE_A), snr=float(SNR), seed=int(SEED))
    px = info['pixel_size']
    DATA = dict(kind='synthetic', star=str(out / 'particles.star'), stack=str(out / 'particles.mrcs'),
                volume=str(out / 'avg_volume.mrc'), mask='', ground_truth=str(out / 'ground_truth.csv'),
                project='synthetic_demo',
                suggest=dict(pixel_size=px, diameter=300.0, resolution=round(2 * px, 3), aperture_index=4, bp_num_psi=1))
    _dataset_file.write_text(json.dumps(DATA, indent=1))
    print(f"✔ {info['n_particles']:,} particles ({info['box']} px, {px:.3g} Å/px) in {info['seconds']:.0f} s → {out}")

In [ ]:
#@title 1.3 · RyR1 tutorial dataset { display-mode: "form" }
#@markdown The RyR1 "great circles" dataset from the [ManifoldEM tutorial](https://github.com/flatironinstitute/ManifoldEM/blob/main/tutorial/README.md)
#@markdown and Ojha *et al.* (2025): 14,491 particles, 1.255 Å/px, diameter 360 Å, resolution 5 Å, aperture index 4.
#@markdown The download is several GB; downsampling (1.5) is recommended on the free tier.
RYR1_URL = "https://users.flatironinstitute.org/~rblackwell/manifold/RyR1GCs_demo.tar.gz"  #@param {type:"string"}
KEEP_ARCHIVE = False  #@param {type:"boolean"}
import tarfile
if DATASET_KIND != 'ryr1':
    print(f'skipped (1.1 selected: {DATASET})')
else:
    dest = mem.data / 'ryr1'
    dest.mkdir(parents=True, exist_ok=True)
    archive = dest / RYR1_URL.rstrip('/').split('/')[-1]
    stars = sorted(dest.rglob('*.star'))
    if not stars:
        downloaded = dest / '.download_complete'
        if not downloaded.exists():          # wget -c resumes an interrupted download
            mem.run(['wget', '-c', '--progress=dot:giga', '-O', archive, RYR1_URL])
            downloaded.touch()
        print('· extracting ...')
        with tarfile.open(archive) as tf:
            tf.extractall(dest, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
        if not KEEP_ARCHIVE:
            archive.unlink()
        stars = sorted(dest.rglob('*.star'))
    stacks = sorted(dest.rglob('*.mrcs'))
    if not stars or not stacks:
        raise RuntimeError(f'no .star / .mrcs found in {dest}')
    star = next((s for s in stars if 'clustRem' in s.name), stars[0])
    stack = next((s for s in stacks if s.stem == star.stem), stacks[0])
    DATA = dict(kind='ryr1', star=str(star), stack=str(stack), volume='', mask='', ground_truth='',
                project='ryr1_tutorial',
                suggest=dict(pixel_size=1.255, diameter=360.0, resolution=5.0, aperture_index=4, downsample=2))
    _dataset_file.write_text(json.dumps(DATA, indent=1))
    print(f'✔ STAR: {star}\n  stack: {stack} ({stack.stat().st_size / 1e9:.1f} GB)')

In [ ]:
#@title 1.4 · Your own data { display-mode: "form" }
#@markdown Paths on Google Drive (mount it in 1.1, e.g. `/content/drive/MyDrive/cryoem/particles.star`), paths on the
#@markdown Colab disk, or `https://` links. Leave the volume and mask empty if you have none.
STAR_FILE = ""  #@param {type:"string"}
PARTICLE_STACK = ""  #@param {type:"string"}
AVERAGE_VOLUME = ""  #@param {type:"string"}
MASK_VOLUME = ""  #@param {type:"string"}
COPY_TO_COLAB_DISK = True  #@param {type:"boolean"}
#@markdown Copying from Drive to the Colab disk makes every step faster and protects the originals (ManifoldEM may
#@markdown rewrite the header of the particle stack).
UPLOAD_FROM_THIS_COMPUTER = False  #@param {type:"boolean"}
#@markdown Upload small files with the browser instead (the STAR file, a mask; stacks of many GB are better put on Drive).
if DATASET_KIND != 'own':
    print(f'skipped (1.1 selected: {DATASET})')
else:
    dest = mem.data / 'own'
    dest.mkdir(parents=True, exist_ok=True)
    if UPLOAD_FROM_THIS_COMPUTER:
        if not IN_COLAB:
            raise RuntimeError('uploading only works inside Colab')
        from google.colab import files
        for name, content in files.upload().items():
            (dest / name).write_bytes(content)
            print(f'uploaded {name} ({len(content) / 1e6:.1f} MB)')

    def fetch(path):
        path = path.strip()
        if not path:
            return ''
        if path.startswith(('http://', 'https://')):
            target = dest / path.split('?')[0].rstrip('/').split('/')[-1]
            if not target.exists():
                mem.run(['wget', '-c', '--progress=dot:giga', '-O', target, path])
            return str(target)
        p = Path(path).expanduser()
        if not p.exists() and (dest / p.name).exists():
            p = dest / p.name                     # an uploaded file given by name
        if not p.exists():
            raise FileNotFoundError(f'{path} not found' + (' (is Google Drive mounted? see 1.1)' if '/drive/' in path else ''))
        if COPY_TO_COLAB_DISK and dest not in p.parents:
            target = dest / p.name
            if not target.exists() or target.stat().st_size != p.stat().st_size:
                print(f'· copying {p.name} ({p.stat().st_size / 1e9:.2f} GB) ...')
                shutil.copy(p, target)
            return str(target)
        return str(p.resolve())

    if not STAR_FILE.strip() or not PARTICLE_STACK.strip():
        raise ValueError('please fill in STAR_FILE and PARTICLE_STACK')
    DATA = dict(kind='own', star=fetch(STAR_FILE), stack=fetch(PARTICLE_STACK), volume=fetch(AVERAGE_VOLUME),
                mask=fetch(MASK_VOLUME), ground_truth='', project='my_project', suggest={})
    _dataset_file.write_text(json.dumps(DATA, indent=1))
    print('✔ ' + '\n  '.join(f'{k}: {DATA[k]}' for k in ('star', 'stack', 'volume', 'mask') if DATA[k]))

In [ ]:
#@title 1.5 · (Optional) Downsample the particles { display-mode: "form" }
#@markdown Fourier-crops every image by `DOWNSAMPLE_BY` (e.g. 336 → 168 px). Every ManifoldEM step becomes roughly
#@markdown 4–8× faster and needs 4× less memory; information beyond the new Nyquist limit (2 × new pixel size) is lost.
#@markdown `0` = automatic (2 for the RyR1 tutorial data, otherwise keep the original size).
DOWNSAMPLE_BY = 0  #@param {type:"number"}
assert DATA, 'choose and prepare a dataset first (1.1–1.4)'
factor = DOWNSAMPLE_BY or DATA['suggest'].get('downsample', 1)
if factor <= 1 or DATA.get('downsampled_by'):
    print('keeping the original images' if factor <= 1 else f"already downsampled by {DATA['downsampled_by']:g}")
else:
    res = mem.bridge('downsample', star=DATA['star'], stack=DATA['stack'], factor=float(factor),
                     out_dir=str(Path(DATA['stack']).parent / f'bin{factor:g}'))
    DATA.update(star=res['star'], stack=res['stack'], downsampled_by=res['scale'])
    sug = DATA['suggest']
    if 'pixel_size' in sug:
        sug['pixel_size'] = round(sug['pixel_size'] * res['scale'], 4)
        sug['resolution'] = max(sug.get('resolution', 0), round(2 * sug['pixel_size'], 3))  # new Nyquist limit
    _dataset_file.write_text(json.dumps(DATA, indent=1))
    print(f"✔ {res['box']} px images → {res['stack']}\n  the pixel size is now {res['scale']:.3g} × the original"
          + (f" = {sug['pixel_size']:g} Å" if 'pixel_size' in sug else ' (use that value in 2.1)'))

In [ ]:
#@title 1.6 · Check the inputs { display-mode: "form" }
#@markdown Reads the STAR file and the stack exactly as ManifoldEM will, and shows a few particles and the
#@markdown distribution of viewing directions and defocus.
assert DATA, 'choose and prepare a dataset first (1.1–1.4)'
INPUT_REPORT = mem.bridge('inspect-inputs', star=DATA['star'], stack=DATA['stack'],
                          out_npz=str(mem.tmp / 'inputs.npz'))
mem.show_inputs(INPUT_REPORT, mem.tmp / 'inputs.npz')
if INPUT_REPORT.get('star_pixel_size') or INPUT_REPORT.get('header_pixel_size'):
    DATA['suggest'].setdefault('pixel_size', INPUT_REPORT.get('star_pixel_size') or INPUT_REPORT.get('header_pixel_size'))
    _dataset_file.write_text(json.dumps(DATA, indent=1))

## 2 · Project

A project is a `params_<name>.toml` file plus an `output/<name>/` folder, both in `/content/manifoldem/work`.

* **Pixel size** in Å (of the images you use, i.e. after downsampling).
* **Object diameter** in Å: the largest dimension of the particle.
* **Resolution** in Å: the resolution of your refinement (at most 2 × the pixel size is meaningful).
* **Aperture index** (1–4, sometimes more): scales the size of the orientation bins. The bin width is
  aperture index × Shannon angle, where Shannon angle = resolution / diameter. Larger bins collect more particles per
  PD but blur the motion with nearby views. It usually takes some trial and error (section 3 shows the effect in seconds).

In [ ]:
#@title 2.1 · Create the project { display-mode: "form" }
PROJECT_NAME = "auto"  #@param {type:"string"}
PIXEL_SIZE_A = 0  #@param {type:"number"}
OBJECT_DIAMETER_A = 0  #@param {type:"number"}
RESOLUTION_A = 0  #@param {type:"number"}
APERTURE_INDEX = 0  #@param {type:"integer"}
#@markdown `auto` / `0` = the value suggested for the selected dataset (or read from the STAR file / stack header).
USE_MASK = True  #@param {type:"boolean"}
OVERWRITE_EXISTING = False  #@param {type:"boolean"}
#@markdown Tick only to start a project of the same name again from scratch: its results are deleted.
import re
assert DATA, 'choose and prepare a dataset first (1.1–1.4)'
sug = DATA.get('suggest', {})
def _pick(value, key, what):
    value = value or sug.get(key)
    if not value:
        raise ValueError(f'please enter {what} (no suggestion available for this dataset)')
    return value
name = DATA['project'] if PROJECT_NAME.strip().lower() in ('', 'auto') else PROJECT_NAME.strip()
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', name) or 'params_' in name:
    raise ValueError('use letters, digits, "_", "-" or "." for the project name (and not "params_")')
px = float(_pick(PIXEL_SIZE_A, 'pixel_size', 'PIXEL_SIZE_A'))
diameter = float(_pick(OBJECT_DIAMETER_A, 'diameter', 'OBJECT_DIAMETER_A'))
resolution = float(_pick(RESOLUTION_A, 'resolution', 'RESOLUTION_A'))
aperture = int(APERTURE_INDEX or sug.get('aperture_index', 1))
if (mem.work / f'params_{name}.toml').exists() and not OVERWRITE_EXISTING:
    raise RuntimeError(f'project {name!r} already exists: re-open it with 2.2, pick another name, or tick OVERWRITE_EXISTING')
cmd = [mem.cli_exe, 'init', '-o', '-p', name, '-a', DATA['star'], '-i', DATA['stack'],
       '-s', px, '-d', diameter, '-r', resolution, '-x', aperture]
if DATA.get('volume'):
    cmd += ['-v', DATA['volume']]
if USE_MASK and DATA.get('mask'):
    cmd += ['-m', DATA['mask']]
mem.run(cmd)
mem.project = name
mem.save_state(dataset=DATA)
mem.refresh()
print(f"✔ project {name}: {px:g} Å/px, diameter {diameter:g} Å, resolution {resolution:g} Å, aperture {aperture} "
      f"→ Shannon angle {resolution / diameter:.4f} rad, bin width {aperture * resolution / diameter:.4f} rad")
print(f"  {mem.S['n_pd']} PDs with the default thresholds — tune them in section 3")

In [ ]:
#@title 2.2 · Re-open a project (after a restart or disconnect) { display-mode: "form" }
#@markdown Skip this cell if you just created the project in 2.1.
PROJECT_TO_OPEN = ""  #@param {type:"string"}
#@markdown Empty = the most recently modified project.
RESTORE_FROM_DRIVE = False  #@param {type:"boolean"}
DRIVE_FOLDER = "/content/drive/MyDrive/ManifoldEM"  #@param {type:"string"}
#@markdown With **RESTORE_FROM_DRIVE**, the newest checkpoint of `PROJECT_TO_OPEN` saved by 10.1 is unpacked first (Drive is mounted if needed).
import tarfile
if RESTORE_FROM_DRIVE:
    if IN_COLAB and not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    pattern = f'{PROJECT_TO_OPEN.strip()}_*.tar.gz' if PROJECT_TO_OPEN.strip() else '*.tar.gz'
    archives = sorted(Path(DRIVE_FOLDER).glob(pattern), key=os.path.getmtime)
    if not archives:
        raise FileNotFoundError(f'no checkpoint {pattern} in {DRIVE_FOLDER}')
    print(f'· unpacking {archives[-1]} ...')
    with tarfile.open(archives[-1]) as tf:
        tf.extractall(mem.root, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
names = mem.projects()
if not names:
    raise RuntimeError(f'no projects in {mem.work}')
name = PROJECT_TO_OPEN.strip() or max(names, key=lambda n: os.path.getmtime(mem.work / f'params_{n}.toml'))
mem.use(name)
DATA = mem.state().get('dataset', DATA)
missing = [f for f in (mem.S['files']['star'], mem.S['files']['stack']) if f and not os.path.exists(f)]
mem.status()
if missing:
    print('⚠️ input files are missing (re-run the dataset cell from section 1 to recreate them): ' + ', '.join(missing))

## 3 · Projection directions

Particles are grouped by viewing direction into bins on a half-sphere. Directions on the other half are mirrored
onto it. Every bin with at least `PRD_THRES_LOW` particles becomes a **projection direction (PD)**, and at most
`PRD_THRES_HIGH` particles per PD are used. Re-run this cell until the map looks reasonable: PDs should cover the
populated views, each with enough particles. Neighbouring PDs form connected groups (*clusters*); each cluster
later needs at least one anchor.

In [ ]:
#@title 3.1 · Bin the particles and set the PD thresholds { display-mode: "form" }
PRD_THRES_LOW = 100  #@param {type:"integer"}
PRD_THRES_HIGH = 2000  #@param {type:"integer"}
APERTURE_INDEX_OVERRIDE = 0  #@param {type:"integer"}
#@markdown `0` keeps the aperture index chosen in 2.1.
TESSELLATION = "keep"  #@param ["keep", "lovisolo_silva", "fibonacci"]
SHOW_3D_SPHERE = False  #@param {type:"boolean"}
DELETE_OUTDATED_RESULTS = True  #@param {type:"boolean"}
#@markdown Re-binning renumbers the PDs, so results of sections 4–8 no longer match and are deleted (untick to keep them).
res = mem.bridge('bin-particles', toml=mem.toml, low=int(PRD_THRES_LOW), high=int(PRD_THRES_HIGH),
                 aperture_index=int(APERTURE_INDEX_OVERRIDE), tess_type='' if TESSELLATION == 'keep' else TESSELLATION)
if res['stale_downstream']:
    if DELETE_OUTDATED_RESULTS:
        mem.bridge('clean', toml=mem.toml, what=['embedding', 'cc', 'landscape', 'trajectory', 'volumes'])
        print('ℹ️  the binning changed: results of sections 4–8 were deleted')
    else:
        print('⚠️ the binning changed but older results were kept: re-run section 4 before going further')
mem.refresh()
mem.show_binning()
if SHOW_3D_SPHERE:
    mem.show_sphere_3d()

## 4 · Manifold embedding of every PD

This is the compute-heavy part, run independently for every PD:

1. **Distances** (`calc-distance`): CTF-aware distances between all images of the PD, after in-plane alignment and filtering.
2. **Embedding** (`manifold-analysis`): a diffusion map of those distances. Its leading eigenvectors (ψ1, ψ2, …) are
   candidate conformational coordinates.
3. **NLSA** (`psi-analysis`): nonlinear Laplacian spectral analysis along each ψ denoises the images and orders them.
4. **Movies** (`nlsa-movie`): one 2D movie per ψ, which you inspect in sections 5–6.

The steps resume where they stopped: if the session dies, re-run 4.1 with `ONLY_MISSING_PDS` ticked.

In [ ]:
#@title 4.0 · Embedding parameters (optional) { display-mode: "form" }
#@markdown The defaults work for most data. Changing a value here invalidates the results of the steps listed in brackets.
NUM_PSI = 8  #@param {type:"slider", min:1, max:8, step:1}
#@markdown Number of eigenvectors ψ analysed and turned into movies [all of 4].
DISTANCE_FILTER = "Butter"  #@param ["Butter", "Gauss"]
FILTER_CUTOFF = 0.5  #@param {type:"number"}
FILTER_ORDER = 8  #@param {type:"integer"}
#@markdown Low-pass filter applied before computing distances; cutoff as a fraction of Nyquist [all of 4].
NLSA_TUNE = 3  #@param {type:"integer"}
#@markdown Diffusion-map kernel tuning: larger = narrower kernel [embedding, NLSA, movies].
CON_ORDER_RANGE = 50  #@param {type:"integer"}
#@markdown NLSA embedding window ("concatenation order") as a fraction of the PD's images [NLSA, movies, section 7].
MOVIE_FPS = 5  #@param {type:"number"}
values = dict(num_psi=int(NUM_PSI), distance_filter_type=DISTANCE_FILTER, distance_filter_cutoff_freq=float(FILTER_CUTOFF),
              distance_filter_order=int(FILTER_ORDER), nlsa_tune=int(NLSA_TUNE), con_order_range=int(CON_ORDER_RANGE),
              nlsa_fps=float(MOVIE_FPS))
changed = mem.bridge('set-params', toml=mem.toml, values=values)['changed']
mem.refresh()
if not changed:
    print('✔ parameters unchanged')
else:
    for k, (old, new) in changed.items():
        print(f'✔ {k}: {old} → {new}')
    first = min({'num_psi': 0, 'distance_filter_type': 0, 'distance_filter_cutoff_freq': 0, 'distance_filter_order': 0,
                 'nlsa_tune': 1, 'con_order_range': 2, 'nlsa_fps': 3}[k] for k in changed)
    redo = ['calc-distance', 'manifold-analysis', 'psi-analysis', 'nlsa-movie'][first:]
    redo = sorted(set(redo) | set(mem.state().get('redo', [])))
    mem.save_state(redo=redo)
    if mem.S['done'].get('calc_distance'):
        print(f"ℹ️  4.1 will recompute all PDs for: {', '.join(redo)}")

In [ ]:
#@title 4.1 · Run distances → embedding → NLSA → movies { display-mode: "form" }
RUN_DISTANCES = True  #@param {type:"boolean"}
RUN_EMBEDDING = True  #@param {type:"boolean"}
RUN_NLSA = True  #@param {type:"boolean"}
RUN_MOVIES = True  #@param {type:"boolean"}
ONLY_MISSING_PDS = True  #@param {type:"boolean"}
#@markdown Skip PDs whose results already exist (resume after an interruption). Untick to recompute everything.
NCPU = 0  #@param {type:"integer"}
#@markdown `0` = all CPUs (automatically reduced for the distance step if memory is short).
ncpu = NCPU or mem.ncpu
S = mem.S
redo = set(mem.state().get('redo', []))
steps = [('calc-distance', RUN_DISTANCES), ('manifold-analysis', RUN_EMBEDDING), ('psi-analysis', RUN_NLSA),
         ('nlsa-movie', RUN_MOVIES)]
dirty = set()            # PDs recomputed by an earlier step in this run: later steps must follow
t_start = time.time()
for step, wanted in steps:
    if not wanted:
        continue
    everything = (not ONLY_MISSING_PDS) or step in redo
    todo = set(range(S['n_pd'])) if everything else set(mem.bridge('incomplete', toml=mem.toml, step=step)['prds']) | dirty
    if not todo:
        print(f'✔ {step}: all {S["n_pd"]} PDs already done')
        continue
    if step == 'nlsa-movie':
        mem.bridge('remove-movies', toml=mem.toml, prds=sorted(todo))   # nlsa-movie skips PDs that have a class_avg.png
    n_workers = ncpu
    if step == 'calc-distance' and np.isfinite(RAM_GB):   # peak memory ≈ 64 bytes × images × pixels² per worker
        need_gb = 64 * min(S['thres_high'], int(mem.A['occupancy'].max())) * S['box'] ** 2 / 1e9
        n_workers = max(1, min(ncpu, int(0.75 * RAM_GB / max(need_gb, 1e-3))))
        if n_workers < ncpu:
            print(f'ℹ️  using {n_workers} worker(s) for the distances to stay within {RAM_GB:.0f} GB of RAM')
    args = [step] + (['--prds', ','.join(map(str, sorted(todo)))] if len(todo) < S['n_pd'] else [])
    print(f'▶ {step}: {len(todo)} PD(s)')
    mem.cli(*args, ncpu=n_workers)
    dirty |= todo
    mem.save_state(redo=sorted(set(mem.state().get('redo', [])) - {step}))
mem.refresh()
mem.status()
print(f'total {(time.time() - t_start) / 60:.1f} min')
if mem.S['done'].get('nlsa_movie'):
    top = int(mem.pd_table().iloc[0].PD)
    display(HTML(f'<b>ψ movies of the most populated PD ({top})</b><br>' + mem.movies_html(top)))

## 5 · Explore the projection directions

For each PD, look for a ψ whose movie shows a **smooth, physically plausible motion** (domains moving, not just
noise flickering or the whole particle shifting or rotating). The class average shows what the PD looks like. The
eigenvalue spectrum shows how many ψ carry signal. The bandwidth plot is the diffusion-map kernel fit (a clear
sigmoid is good). The embedding ψ1–ψ2 is coloured by in-plane rotation: if the colour follows the curve, the "motion"
is just residual rotation.

In [ ]:
#@title 5.1 · Overview of all PDs { display-mode: "form" }
SHOW_TOP = 15  #@param {type:"integer"}
mem.refresh()
display(HTML(mem.pd_table(top=SHOW_TOP or None).to_html(index=False)))
mem.show_anchor_map(title='PDs on the half-sphere')

In [ ]:
#@title 5.2 · PD browser { display-mode: "form" }
SHOW_PD = 0  #@param {type:"integer"}
#@markdown `0` = interactive browser (slider + list sorted by particle count); a number shows that PD only.
if SHOW_PD:
    mem.show_pd(int(SHOW_PD))
else:
    mem.pd_browser()

## 6 · Anchors: tell ManifoldEM which motion to follow

ManifoldEM propagates *one* conformational coordinate across all PDs with optical flow and belief propagation. It
needs **anchors**: PDs where you state which ψ movie shows the motion of interest and in which direction it runs.

* Give **at least one anchor per cluster** (clusters are the colours on the map). PDs in a cluster without an
  anchor get no coordinate.
* The **sense** must be consistent: if a domain moves *up* in your `FWD` anchor's movie, every other `FWD` anchor
  must show the same motion running the same way. Mark an anchor `REV` if its movie shows the motion backwards.
* Good anchors have many particles and a clean, unambiguous movie. More anchors make the propagation more reliable.
* You can also **remove** PDs whose movies are junk; they are excluded from the landscape and the volumes.

The RyR1 tutorial's anchors (0-based PDs 52 FWD and 45 REV in the tutorial text) are `53:1:FWD, 46:1:REV` here,
provided your binning matches the tutorial's.

In [ ]:
#@title 6.1 · Candidate anchors: the most populated PDs of every cluster { display-mode: "form" }
PDS_PER_CLUSTER = 2  #@param {type:"integer"}
PSI_TO_SHOW = "1, 2, 3"  #@param {type:"string"}
psis = [int(p) for p in PSI_TO_SHOW.replace(' ', '').split(',') if p]
mem.anchor_candidates(per_cluster=int(PDS_PER_CLUSTER), psis=[p for p in psis if 1 <= p <= mem.S['num_psi']])

In [ ]:
#@title 6.2 · Set anchors and removed PDs { display-mode: "form" }
ANCHORS = "auto"  #@param {type:"string"}
#@markdown Comma-separated `PD:ψ:sense` entries, e.g. `53:1:FWD, 46:1:REV` (PD and ψ numbers as shown in this notebook).
#@markdown `auto` = the most populated PD of every cluster, ψ1, `FWD`; check its movies before trusting it on real data.
REMOVE_PDS = ""  #@param {type:"string"}
#@markdown PDs to exclude, e.g. `7, 12-14`. Leave empty to keep all.
anchors = mem.parse_anchors(ANCHORS)
trash = mem.parse_pd_list(REMOVE_PDS)
res = mem.bridge('set-anchors', toml=mem.toml, anchors=anchors, trash=trash)
mem.refresh()
mem.show_anchor_map()
print('anchors: ' + ', '.join(f'PD {a + 1} ψ{k} {s}' for a, k, s in anchors) + (f' · removed: {len(trash)} PD(s)' if trash else ''))
for comp in res['unanchored_components']:
    print(f'⚠️ no anchor among PDs {comp}: they will get no conformational coordinate')

## 7 · Conformational coordinate and energy landscape

* **7.1** finds, for every PD, the ψ and the sense that match the anchors. It computes optical flow between the
  movies of neighbouring PDs and runs belief propagation over the graph of PDs.
* **7.2** re-runs NLSA with those choices, places every image on the common coordinate τ ∈ [0, 1] and histograms the
  occupancy. Its negative logarithm is the free-energy profile, in units of k<sub>B</sub>T.
* **7.3** sorts the images into `STATES` bins along τ and writes one particle stack + STAR file per state, ready for 3D
  reconstruction.

In [ ]:
#@title 7.1 · Find the conformational coordinate (belief propagation) { display-mode: "form" }
BP_NUM_PSI = "auto"  #@param ["auto", "all", "1", "2", "3", "4", "5", "6", "7", "8"]
#@markdown How many eigenvectors belief propagation may choose from in each PD: `all` is ManifoldEM's standard
#@markdown behaviour; `1` keeps ψ1 everywhere and only propagates the sense (robust when ψ1 clearly shows the motion in
#@markdown most PDs). `auto` = `1` for the synthetic demo, `all` otherwise.
if BP_NUM_PSI == 'auto':
    bp_num_psi = int(mem.state().get('dataset', {}).get('suggest', {}).get('bp_num_psi', 0))
else:
    bp_num_psi = 0 if BP_NUM_PSI == 'all' else int(BP_NUM_PSI)     # 0 = all of ψ1..ψ<NUM_PSI>
res = mem.bridge('find-ccs', toml=mem.toml, ncpu=mem.ncpu, bp_num_psi=bp_num_psi)
mem.refresh()
print(f"✔ {res['n_assigned']} of {res['n_pd']} PDs assigned in {res['seconds']:.0f} s"
      f" (belief propagation over ψ1–ψ{res['bp_num_psi']})")
if res['anchors_shifted']:
    print('ℹ️  this ManifoldEM version reads Anchor.CC as 0-based during belief propagation; the notebook compensated, '
          'so your anchors refer to the ψ numbers shown here')
mem.show_cc();

In [ ]:
#@title 7.2 · Probability and free-energy landscape { display-mode: "form" }
STATES = 50  #@param {type:"integer"}
#@markdown Number of bins along the coordinate (also the number of volumes in section 8).
if mem.bridge('set-params', toml=mem.toml, values=dict(states_per_coord=int(STATES)))['changed']:
    mem.bridge('clean', toml=mem.toml, what=['trajectory', 'volumes'])
mem.cli('probability-landscape')
mem.refresh()
gt = mem.state().get('dataset', {}).get('ground_truth')
GT_ROWS = mem.bridge('ground-truth', toml=mem.toml, gt_csv=gt)['rows'] if gt and os.path.exists(gt) else None
mem.show_landscape(GT_ROWS);

In [ ]:
#@title 7.3 · Write per-state particle stacks (3D trajectories) { display-mode: "form" }
PATH_WIDTH = 1  #@param {type:"slider", min:1, max:5, step:1}
#@markdown Number of neighbouring bins pooled into each state (more images per volume, less resolution along τ).
mem.bridge('set-params', toml=mem.toml, values=dict(width_1D=int(PATH_WIDTH)))
mem.bridge('clean', toml=mem.toml, what=['trajectory', 'volumes'])   # stale states would be mixed in otherwise
mem.cli('trajectory')
mem.refresh()
stars = sorted(glob.glob(os.path.join(mem.P['bin_dir'], '*.star')))
counts = [sum(1 for l in open(s) if '@' in l) for s in stars]
print(f'✔ {len(stars)} states, {min(counts)}–{max(counts)} images each → {mem.P["bin_dir"]}')

## 8 · 3D volumes along the coordinate

ManifoldEM's `utility mrcs2mrc` reconstructs every state with RELION's `relion_reconstruct`. Colab can install a
CPU build of RELION from Ubuntu's package archive in about a minute. The built-in NumPy reconstructor (direct
Fourier inversion, validated against RELION) is a fallback that needs no installation.

In [ ]:
#@title 8.1 · Reconstruct one volume per state { display-mode: "form" }
METHOD = "RELION relion_reconstruct (apt install, ~1 min)"  #@param ["RELION relion_reconstruct (apt install, ~1 min)", "NumPy (built in, no installation)"]
stars = sorted(glob.glob(os.path.join(mem.P['bin_dir'], '*.star')))
assert stars, 'run 7.3 first'
mem.bridge('clean', toml=mem.toml, what=['volumes'])
box = mem.S['box']
per_job_gb = 0.15 + 4.7 * (box / 256) ** 3           # measured: ~0.6 GB at 128 px, ~4.7 GB at 256 px
n_jobs = max(1, min(mem.ncpu, int(0.75 * RAM_GB / per_job_gb))) if np.isfinite(RAM_GB) else 1
use_relion = METHOD.startswith('RELION')
if use_relion and not shutil.which('relion_reconstruct'):
    print('· installing RELION from the Ubuntu archive ...')
    try:
        apt = [] if os.geteuid() == 0 else ['sudo']
        mem.run(apt + ['apt-get', '-qq', 'update'], echo=False)
        mem.run(apt + ['env', 'DEBIAN_FRONTEND=noninteractive', 'apt-get', '-qq', 'install', '-y', 'relion'], echo=False)
    except Exception as exc:
        print(f'⚠️ RELION could not be installed ({exc}); using the NumPy reconstructor instead')
        use_relion = False
if use_relion:
    test = mem.bridge('relion-selftest')
    print(f"✔ relion_reconstruct self-test: correlation with the phantom {test['cc']:.4f}")
    mem.cli('utility', 'mrcs2mrc', ncpu=n_jobs)
    missing = [s for s in stars if not os.path.exists(os.path.join(mem.P['vols_dir'], os.path.basename(s)[:-5] + '.mrc'))]
    for s in missing:   # manifold-cli hides RELION's messages; retry visibly
        print(f'· retrying {os.path.basename(s)}')
        mem.run(['relion_reconstruct', '--i', os.path.basename(s), '--o',
                 os.path.join(mem.P['vols_dir'], os.path.basename(s)[:-5] + '.mrc')], cwd=mem.P['bin_dir'], check=False)
else:
    mem.bridge('reconstruct', toml=mem.toml, ncpu=n_jobs, pad=2 if box <= 256 else 1)
mem.refresh()
VOLUME_FILES = mem.volume_files(mem.P['vols_dir'])
print(f'✔ {len(VOLUME_FILES)} of {len(stars)} volumes in {mem.P["vols_dir"]}')

In [ ]:
#@title 8.2 · Denoise the volumes { display-mode: "form" }
GAUSSIAN_SIGMA_VOXELS = 1.0  #@param {type:"number"}
SVD_COMPONENTS = 0  #@param {type:"integer"}
#@markdown `SVD_COMPONENTS > 0` keeps only the leading components of the whole series of volumes (a strong denoiser for
#@markdown smooth motions; try 3–5). Results go to `postproc/denoise_notebook/`; the raw volumes are kept.
#@markdown (ManifoldEM's `utility denoise` only filters the first and last 5 states.)
VOLUME_FILES = mem.volume_files(mem.P['vols_dir'])
assert VOLUME_FILES, 'run 8.1 first'
DENOISED_DIR = os.path.join(mem.P['out_dir'], 'postproc', 'denoise_notebook')
DENOISED_FILES = denoise_volumes(VOLUME_FILES, DENOISED_DIR, sigma=float(GAUSSIAN_SIGMA_VOXELS), svd_rank=int(SVD_COMPONENTS))
print(f'✔ {len(DENOISED_FILES)} denoised volumes → {DENOISED_DIR}')

In [ ]:
#@title 8.3 · Watch the conformational change { display-mode: "form" }
SHOW = "denoised"  #@param ["denoised", "raw"]
FLIP_CONTRAST = "auto"  #@param ["auto", "yes", "no"]
#@markdown ManifoldEM's trajectory images have inverted contrast, so volumes often come out negative; `auto` fixes that.
ISO_LEVEL_SIGMA = 3.0  #@param {type:"number"}
ISOSURFACE_STATES = 10  #@param {type:"integer"}
denoised_dir = os.path.join(mem.P['out_dir'], 'postproc', 'denoise_notebook')
folder = denoised_dir if SHOW == 'denoised' and mem.volume_files(denoised_dir) else mem.P['vols_dir']
if SHOW == 'denoised' and folder != denoised_dir:
    print('no denoised volumes yet (8.2): showing the raw ones')
files = mem.volume_files(folder)
assert files, 'run 8.1 (and 8.2) first'
flip = needs_flip(files, FLIP_CONTRAST)
proj = volume_projections(files, flip)
show_volume_strip(proj, title=f'{len(files)} states along the conformational coordinate ({SHOW} volumes)')
movie_dir = os.path.join(mem.P['out_dir'], 'postproc', 'notebook')
os.makedirs(movie_dir, exist_ok=True)
volume_movie(proj, os.path.join(movie_dir, 'trajectory_projections.gif'))
states = sorted(set(np.linspace(0, len(files) - 1, max(2, min(ISOSURFACE_STATES, len(files)))).round().astype(int)))
isosurface_movie(files, states, flip=flip, sigma=float(ISO_LEVEL_SIGMA), step=2 if mem.S['box'] <= 160 else 3)
print(f'Tip — in ChimeraX: open {folder}/*.mrc vseries true')

## 9 · Python API (advanced)

Everything ManifoldEM computes is available through its documented `data_store` API (see `help(data_store)` and the
main README). The API lives in ManifoldEM's own environment, so start a cell with **`%%manifoldem`** to run it there,
with `np`, `plt`, `params`, `data_store`, `prds` (the projection directions), `Anchor` and `Sense` already loaded.
Printed output and matplotlib figures appear below the cell. Indices in the API are **0-based** (PD *n* in this
notebook is index *n − 1*).

In [ ]:
%%manifoldem
top = int(np.argmax(prds.occupancy))                  # 0-based index of the most populated PD
prd = data_store.get_prd_data(top)
print(f'PD {top + 1} (index {top})')
print(prd)
print('embedding keys:', sorted(prd.psi_data.keys()))
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, img, title in zip(ax, (prd.raw_images[0], prd.transformed_images[0], prd.ctf_images[0]),
                         ('raw image', 'filtered + aligned', 'CTF')):
    a.imshow(img, cmap='gray'); a.set_title(title); a.axis('off')

## 10 · Save and download

The Colab disk is wiped when the runtime is recycled. **10.1** packs the project (and optionally the input data)
into one archive on Google Drive, which **2.2** can restore in a later session. **10.2** downloads the results to
your computer.

In [ ]:
#@title 10.1 · Save a checkpoint to Google Drive { display-mode: "form" }
DRIVE_FOLDER = "/content/drive/MyDrive/ManifoldEM"  #@param {type:"string"}
INCLUDE_INPUT_DATA = False  #@param {type:"boolean"}
#@markdown Include the STAR file and particle stack, if they are on the Colab disk (needed to continue the analysis later).
if IN_COLAB and not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
members = [f'work/params_{mem.project}.toml', f'work/output/{mem.project}', f'work/.notebook/{mem.project}.json']
if INCLUDE_INPUT_DATA:
    for f in {mem.S['files'][k] for k in ('star', 'stack', 'volume', 'mask') if mem.S['files'][k]}:
        if Path(f).resolve().is_relative_to(mem.root.resolve()):
            members.append(str(Path(f).resolve().relative_to(mem.root.resolve())))
        else:
            print(f'not included (outside {mem.root}): {f}')
members = [m for m in members if (mem.root / m).exists()]
archive = Path(DRIVE_FOLDER) / f'{mem.project}_{time.strftime("%Y%m%d-%H%M%S")}.tar.gz'
tar = ['tar', '-I', 'pigz'] if shutil.which('pigz') else ['tar', '-z']
mem.run(tar + ['-cf', archive, '-C', mem.root] + members)
print(f'✔ {archive} ({archive.stat().st_size / 1e9:.2f} GB)')

In [ ]:
#@title 10.2 · Download the results { display-mode: "form" }
WHAT = "results (volumes, landscape, tables, movies)"  #@param ["results (volumes, landscape, tables, movies)", "whole project (large)"]
import zipfile, pandas as pd
out = Path(mem.P['out_dir'])
export = out / 'postproc' / 'notebook'
export.mkdir(parents=True, exist_ok=True)
mem.pd_table().to_csv(export / 'projection_directions.csv', index=False)
if os.path.exists(mem.P['CC_file']):
    cc = _read_pickle(mem.P['CC_file'])
    pd.DataFrame(dict(PD=np.arange(1, mem.S['n_pd'] + 1), psi=cc['psinums'][0] + 1, sense=cc['senses'][0])).to_csv(
        export / 'conformational_coordinates.csv', index=False)   # psi 0 = unassigned
if os.path.exists(mem.P['traj']):
    h = np.asarray(_read_pickle(mem.P['traj'])['hUn'], dtype=float)
    with np.errstate(divide='ignore'):
        pd.DataFrame(dict(state=np.arange(1, len(h) + 1), particles=h, free_energy_kT=-np.log(h / h.max()))).to_csv(
            export / 'landscape.csv', index=False)
zip_path = mem.root / f'{mem.project}_{"results" if WHAT.startswith("results") else "project"}.zip'
if WHAT.startswith('results'):
    picks = [mem.toml] + [p for p in out.rglob('*') if p.is_file() and (
        'postproc' in p.parts or p.parent.name == 'traj' or p.name in ('comp_psi_sense_nodes.txt', 'CC_file.pkl'))]
else:
    picks = [mem.toml] + [p for p in out.rglob('*') if p.is_file()]
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in picks:
        z.write(p, Path(p).relative_to(mem.work))
print(f'✔ {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))

## 11 · Troubleshooting

| Symptom | What to do |
|---|---|
| A cell says *run … first* | Run the cells in order. After a restart, run section 0 and then 2.2. |
| `NameError: mem` | Section 0 has not been run in this session. |
| ManifoldEM reports a missing STAR column | See 1.6. Export the STAR file again from RELION / cryoSPARC with angles, defocus and optics. |
| *the STAR file has N particles but the stack has M images* | ManifoldEM pairs STAR row *i* with stack image *i*. Write a single stack in STAR order (`relion_stack_create`). |
| Very few PDs | Lower `PRD_THRES_LOW` or increase the aperture index (3.1). Keep `PRD_THRES_LOW ≥ 2 × CON_ORDER_RANGE`. |
| Session crashes ("used all available RAM") | Downsample (1.5), lower `PRD_THRES_HIGH` (3.1) and/or set `NCPU` to 1 in 4.1, then resume with `ONLY_MISSING_PDS`. |
| The session disconnected during section 4 | Run section 0, 2.2, then 4.1 with `ONLY_MISSING_PDS` ticked: finished PDs are kept. |
| Many PDs unassigned after 7.1 | Add an anchor to every cluster (6.2); remove junk PDs; try `BP_NUM_PSI = 1` if ψ1 shows the motion almost everywhere. |
| Landscape looks like noise | Check the anchors' senses: one inverted anchor scrambles the coordinate. Look at more PDs in 5.2. |
| RELION fails to install | 8.1 falls back to the NumPy reconstructor automatically. |
| Volumes look inverted or empty | `FLIP_CONTRAST` / `ISO_LEVEL_SIGMA` in 8.3; ChimeraX gives full control (`open *.mrc vseries true`). |

**What the notebook does differently from typing the CLI by hand** (all ManifoldEM v0.4.0 behaviour; check
again on newer versions):
* ManifoldEM runs in a separate Python 3.12 environment with NumPy < 2 (its `nptyping` dependency and some code do not
  support NumPy 2).
* Parameters are written into the project's `.toml` (`manifold-cli <step> --param` flags other than for `init` and
  `threshold` are not persistent).
* Belief propagation in v0.4.0 reads `Anchor.CC` as 0-based while the GUI and the tutorial use 1-based ψ numbers; 7.1
  compensates, so `Anchor(CC=1)` means ψ1 here as in the GUI.
* `find-ccs` with no anchors silently does nothing. 6.2 / 7.1 check for that, and they warn about clusters without
  an anchor.
* `utility mrcs2mrc` hides RELION errors, so 8.1 checks that every volume was written. `utility denoise` only
  filters the first and last states, so 8.2 denoises all of them and keeps the pixel size.

**Links:** [ManifoldEM on GitHub](https://github.com/flatironinstitute/ManifoldEM) ·
[tutorial](https://github.com/flatironinstitute/ManifoldEM/blob/main/tutorial/README.md) ·
[issues](https://github.com/flatironinstitute/ManifoldEM/issues) ·
Dashti *et al.*, PNAS 2014 ([doi](https://doi.org/10.1073/pnas.1419276111)) ·
Dashti *et al.*, Nat. Commun. 2020 ([doi](https://doi.org/10.1038/s41467-020-18403-x)) ·
Seitz *et al.*, IEEE TCI 2022 ([doi](https://doi.org/10.1109/TCI.2022.3174801)) ·
Ojha *et al.*, Acta Cryst. D 2025 ([doi](https://doi.org/10.1107/S2059798325001469))